# Biohub Cell Tracking — Ultra-Deep Phase-Wise Notebook
### Data Model → EDA → Classical Baseline → GPU Deep-Learning Enhanced Pipeline → Submission

This notebook is organized into explicit, independently re-runnable **phases**. Every phase persists its
outputs (figures as PNG, tables as CSV, models as PT) into `/kaggle/working/outputs/...` so the whole run
can be reviewed and zipped at the end without re-executing earlier phases.

| # | Phase | Produces |
|---|---|---|
| 0 | Environment & GPU bring-up | GPU diagnostics table |
| 1 | Data contract & configuration | `CFG` dataclass, paths, run-mode switches |
| 2 | Metadata EDA | sample / embryo / array-geometry tables + plots |
| 3 | Ground-truth graph EDA | sparse-graph stats, motion & division geometry + plots |
| 4 | Classical baseline detector | Otsu / percentile peak detection + visual QA |
| 5 | GPU deep-learning detector | 3D heatmap-regression CNN trained on GT centroids |
| 6 | Detector comparison & calibration | classical vs. DL count / recall sweeps |
| 7 | Tracking / linking | Hungarian assignment + motion-gated linking + divisions |
| 8 | Local graph proxy scoring | sparse node match + edge / division proxy metrics |
| 9 | Submission build & audit | streaming CSV writer + schema / value audit |
| 10 | Export & packaging | zips every artifact for download, prints sanity-check summary |

Target: **40+ saved figures**, **20+ distinct analyses/tables**. The notebook degrades gracefully (skips DL
training, falls back to classical-only submission) if no GPU/torch is available, and degrades gracefully if
the Kaggle dataset isn't mounted yet (so the structure can be sanity-checked anywhere), while being written
to fully exploit Kaggle's free GPU when it is present.

**Before running:** in the right sidebar choose *Accelerator → GPU T4 x2 (or P100)*, attach the
`biohub-cell-tracking-during-development` dataset, then *Run All*.

## Phase 0 — Environment & GPU Bring-up

In [114]:
import os, sys, json, time, warnings, shutil, zipfile, gc, math
from pathlib import Path
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')
print('Python:', sys.version.split()[0])

# Make sure core scientific packages exist; Kaggle base image has all of these.
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter, center_of_mass
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from IPython.display import display

try:
    import zarr
except ImportError:
    zarr = None
try:
    import blosc2
except ImportError:
    blosc2 = None
try:
    from skimage.feature import peak_local_max
except Exception:
    peak_local_max = None
try:
    from skimage.filters import threshold_otsu
except Exception:
    threshold_otsu = None

# ---- Output directory scaffold: created up-front so every phase can write into it ----
OUT_ROOT  = Path('/kaggle/working/outputs')
FIG_DIR   = OUT_ROOT / 'figures'
TAB_DIR   = OUT_ROOT / 'tables'
MODEL_DIR = OUT_ROOT / 'models'
SUB_DIR   = OUT_ROOT / 'submission'
LOG_DIR   = OUT_ROOT / 'logs'
for d in (OUT_ROOT, FIG_DIR, TAB_DIR, MODEL_DIR, SUB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

FIGURE_REGISTRY = []   # (phase, name, path)
TABLE_REGISTRY  = []   # (phase, name, path)

def save_fig(fig, phase, name, dpi=130):
    '''Save + register a matplotlib figure under outputs/figures.'''
    path = FIG_DIR / f'{phase}_{name}.png'
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    FIGURE_REGISTRY.append((phase, name, str(path)))
    plt.close(fig)
    return path

def save_table(df, phase, name, index=False):
    '''Save + register a DataFrame under outputs/tables.'''
    path = TAB_DIR / f'{phase}_{name}.csv'
    df.to_csv(path, index=index)
    TABLE_REGISTRY.append((phase, name, str(path)))
    return path

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})
COLORS = {'blue': '#3366AA', 'green': '#2E8B57', 'red': '#B4473A', 'purple': '#6A4C93',
          'orange': '#C77C2E', 'gray': '#6B7280', 'light': '#D8DEE9', 'teal': '#1B998B'}
print('Output scaffold ready at', OUT_ROOT)

Python: 3.12.13
Output scaffold ready at /kaggle/working/outputs


In [115]:
# ---- GPU / torch diagnostics ----
gpu_rows = []
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
    cuda_ok = torch.cuda.is_available()
    gpu_rows.append(['torch_version', torch.__version__])
    gpu_rows.append(['cuda_available', cuda_ok])
    if cuda_ok:
        gpu_rows.append(['device_name', torch.cuda.get_device_name(0)])
        gpu_rows.append(['device_count', torch.cuda.device_count()])
        props = torch.cuda.get_device_properties(0)
        gpu_rows.append(['total_memory_GB', round(props.total_memory / 1e9, 2)])
    DEVICE = torch.device('cuda' if cuda_ok else 'cpu')
except Exception as exc:
    TORCH_OK = False
    DEVICE = None
    gpu_rows.append(['torch_import_error', str(exc)])

GPU_AVAILABLE = TORCH_OK and (DEVICE is not None) and (DEVICE.type == 'cuda')
gpu_table = pd.DataFrame(gpu_rows, columns=['item', 'value'])
display(gpu_table)
save_table(gpu_table, 'phase0', 'gpu_diagnostics')

if GPU_AVAILABLE:
    print('GPU ENABLED -> Phase 5 (deep-learning detector) will train on GPU.')
else:
    print('No GPU/torch detected -> Phase 5 will run a short CPU-safe fallback, or be skipped automatically.')
    print('On Kaggle: right sidebar -> Accelerator -> GPU T4 x2 / P100 -> Save & re-run.')

,item,value
0,torch_version,2.10.0+cu128
1,cuda_available,True
2,device_name,Tesla T4
3,device_count,2
4,total_memory_GB,15.64


GPU ENABLED -> Phase 5 (deep-learning detector) will train on GPU.


In [116]:
# Install SHAP and LIME
!pip install -q --upgrade shap lime

# Import to verify installation
import shap
import lime
import lime.lime_tabular

print("SHAP Version:", shap.__version__)
print("LIME installed successfully!")

SHAP Version: 0.52.0
LIME installed successfully!


## Phase 1 — Data Contract & Configuration

**Image volumes**: Zarr v3 at path `0/`, shape `(T, Z, Y, X)` (typ. `(100, 64, 256, 256)`, `uint16`),
chunked one timepoint per chunk at `0/c/{t}/0/0/0`, voxel scale `z=1.625, y=x=0.40625 um`.

**Ground truth (`.geff`)**: `nodes/ids`, `nodes/props/{t,z,y,x}/values`, `edges/ids` (N,2). Sparse — not
every cell in every frame is labelled. `estimated_number_of_nodes` in `zarr.json` anchors expected density.

**Output**: streaming CSV with header `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`, one row per
node and one row per edge, sentinel `-1` for unused fields.

In [117]:
@dataclass
class CONFIG:
    # ---- paths / run-mode ----
    SUBMIT_MODE: bool = False
    SCALE: tuple = (1.625, 0.40625, 0.40625)
    MATCH_GATE_UM: float = 7.0
    TEST_DIR: str = '/kaggle/input/biohub-cell-tracking-during-development/test'
    TRAIN_DIR: str = '/kaggle/input/biohub-cell-tracking-during-development/train'

    # ---- classical detector ----
    DETECTION_MODE: str = 'classical'
    XY_DS: int = 4
    SMOOTH_SIGMA: tuple = (1.0, 1.0, 1.0)
    MIN_PEAK_DIST: int = 3
    THRESH_REL: float = 0.18
    THRESH_HI_PERCENTILE: float = 99.8
    USE_CENTROID_REFINEMENT: bool = True
    REFINE_RADIUS_Z: int = 2
    REFINE_RADIUS_YX: int = 6
    USE_PHYSICAL_NMS: bool = True
    NMS_RADIUS_UM: float = 4.0
    USE_BORDER_FILTER: bool = True
    BORDER_KEEP_QUANTILE: float = 0.02
    USE_COUNT_STABILIZER: bool = True

    # ---- deep-learning detector ----
    USE_DL_DETECTOR: bool = True
    DL_PATCH_Z: int = 32
    DL_PATCH_Y: int = 128
    DL_PATCH_X: int = 128
    DL_HEATMAP_SIGMA_VOX: float = 1.6
    DL_EPOCHS: int = 6
    DL_BATCH_SIZE: int = 4
    DL_LR: float = 1e-3
    DL_TRAIN_SAMPLES: int = 6
    DL_PATCHES_PER_SAMPLE: int = 24
    DL_PEAK_THRESH: float = 0.45
    DL_MIN_PEAK_DIST: int = 3

    # ---- linking / tracking ----
    MAX_LINK_DIST_UM: float = 10.5
    TIGHT_LINK_PROFILE: bool = False
    DETECT_DIVISIONS: bool = True
    DIV_PARENT_DIST_UM: float = 9.0
    DIV_SISTER_DIST_UM: float = 9.0
    DIV_USE_MIDPOINT: bool = True
    DIV_MIDPOINT_DIST: float = 6.0
    PRUNE_ISOLATED_NODES: bool = False

    # ---- EDA / dev switches ----
    EDA_SAMPLE_LIMIT: int = 8
    GT_EDA_SAMPLE_LIMIT: int = 6
    RUN_BASIC_EDA: bool = True
    RUN_GT_EDA: bool = True
    RUN_VISUAL_EDA: bool = True
    RUN_LOCAL_PROXY: bool = True
    LOCAL_PROXY_SAMPLE_LIMIT: int = 6
    SUBMISSION_LIMIT: 'int | None' = None
    RUN_SUBMISSION_DISPLAY: bool = True
    RUN_SUBMISSION_PLOTS: bool = True

CFG = CONFIG()
print('Config initialised.')

Config initialised.


In [118]:
def find_data_root():
    local_root = os.environ.get('BIOHUB_DATA_ROOT')
    candidates = [Path(local_root)] if local_root else []
    candidates.extend([
        Path(CFG.TEST_DIR).parent,
        Path('/kaggle/input/biohub-cell-tracking-during-development'),
        Path.cwd(), Path.cwd().parent,
        Path.cwd() / 'data', Path.cwd().parent / 'data',
    ])
    for root in candidates:
        if (root / 'test').is_dir() or (root / 'train').is_dir():
            return root
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for test_dir in kaggle_input.glob('**/test'):
            if list(test_dir.glob('*.zarr')):
                return test_dir.parent
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    print('WARNING: no train/ or test/ folder found under /kaggle/input. '
          'Attach the competition dataset and re-run. The notebook will continue '
          'in a degraded "structure-only" mode so it can still be syntax/sanity checked.')
    TRAIN_DIR = TEST_DIR = SAMPLE_SUBMISSION_PATH = None
else:
    TRAIN_DIR = DATA_ROOT / 'train'
    TEST_DIR  = DATA_ROOT / 'test'
    SAMPLE_SUBMISSION_PATH = DATA_ROOT / 'sample_submission.csv'
    print('DATA_ROOT  :', DATA_ROOT)
    print('TRAIN_DIR  :', TRAIN_DIR, '| exists:', TRAIN_DIR.is_dir())
    print('TEST_DIR   :', TEST_DIR, '| exists:', TEST_DIR.is_dir())

DATA_ROOT  : /kaggle/input/competitions/biohub-cell-tracking-during-development
TRAIN_DIR  : /kaggle/input/competitions/biohub-cell-tracking-during-development/train | exists: True
TEST_DIR   : /kaggle/input/competitions/biohub-cell-tracking-during-development/test | exists: True


In [119]:
problem_contract = pd.DataFrame([
    ['task', '3D+time cell centroid detection, tracking, and division linking'],
    ['image array', 'Zarr v3, path 0/, shape (T, Z, Y, X)'],
    ['chunk layout', 'one timepoint per chunk: 0/c/{t}/0/0/0'],
    ['voxel scale', f'z={CFG.SCALE[0]}, y=x={CFG.SCALE[1]} um/voxel'],
    ['matching radius', f'{CFG.MATCH_GATE_UM} um physical distance'],
    ['output schema', 'CSV graph rows: node + edge, header id,dataset,row_type,node_id,t,z,y,x,source_id,target_id'],
    ['detectors available', 'classical (Otsu/percentile peaks) + DL (3D heatmap-regression CNN, GPU)'],
])
problem_contract.columns = ['item', 'value']
display(problem_contract)
save_table(problem_contract, 'phase1', 'problem_contract')

config_table = pd.DataFrame(list(asdict(CFG).items()), columns=['parameter', 'value'])
display(config_table)
save_table(config_table, 'phase1', 'config_snapshot')

,item,value
0,task,"3D+time cell centroid detection, tracking, and..."
1,image array,"Zarr v3, path 0/, shape (T, Z, Y, X)"
2,chunk layout,one timepoint per chunk: 0/c/{t}/0/0/0
3,voxel scale,"z=1.625, y=x=0.40625 um/voxel"
4,matching radius,7.0 um physical distance
5,output schema,"CSV graph rows: node + edge, header id,dataset..."
6,detectors available,classical (Otsu/percentile peaks) + DL (3D hea...


,parameter,value
0,SUBMIT_MODE,False
1,SCALE,"(1.625, 0.40625, 0.40625)"
2,MATCH_GATE_UM,7.0
3,TEST_DIR,/kaggle/input/biohub-cell-tracking-during-deve...
4,TRAIN_DIR,/kaggle/input/biohub-cell-tracking-during-deve...
5,DETECTION_MODE,classical
6,XY_DS,4
7,SMOOTH_SIGMA,"(1.0, 1.0, 1.0)"
8,MIN_PEAK_DIST,3
9,THRESH_REL,0.18


PosixPath('/kaggle/working/outputs/tables/phase1_config_snapshot.csv')

## Phase 2 — Metadata EDA

Sample/embryo counts, array geometry, and `estimated_number_of_nodes` are inspected first because they
directly anchor later modelling decisions (embryo-disjoint validation, working-grid feasibility, the
detection-density ratio rho = N_hat / N_est).

In [120]:
def list_zarr_names(split_dir):
    if split_dir is None or not Path(split_dir).is_dir():
        return []
    return sorted(p.name[:-5] for p in Path(split_dir).glob('*.zarr'))

def embryo_id(name):
    return name.split('_')[0]

def select_diverse_samples(names, limit):
    selected, seen = [], set()
    for name in names:
        e = embryo_id(name)
        if e in seen:
            continue
        selected.append(name); seen.add(e)
        if len(selected) >= limit:
            return selected
    for name in names:
        if name not in selected:
            selected.append(name)
            if len(selected) >= limit:
                break
    return selected

def read_zarr_meta(zarr_path):
    meta_path = zarr_path / '0' / 'zarr.json'
    if not meta_path.exists():
        return None
    with open(meta_path) as f:
        meta = json.load(f)
    shape = tuple(meta.get('shape', []))
    dtype = meta.get('data_type') or meta.get('dtype')
    chunk_shape = None
    cg = meta.get('chunk_grid', {}).get('configuration', {})
    if 'chunk_shape' in cg:
        chunk_shape = cg['chunk_shape']
    return {'shape': shape, 'dtype': dtype, 'chunk_shape': chunk_shape}

def geff_estimated_nodes(geff_path):
    meta_path = geff_path / 'zarr.json'
    if not meta_path.exists():
        return None
    try:
        with open(meta_path) as f:
            meta = json.load(f)
        if 'estimated_number_of_nodes' in meta:
            return meta['estimated_number_of_nodes']
        attrs = meta.get('attributes', {})
        return attrs.get('estimated_number_of_nodes')
    except Exception:
        return None

def scan_split(split_dir, limit):
    names = list_zarr_names(split_dir)
    sampled = select_diverse_samples(names, limit) if names else []
    rows = []
    for name in sampled:
        zpath = Path(split_dir) / f'{name}.zarr'
        meta = read_zarr_meta(zpath)
        gpath = Path(split_dir) / f'{name}.geff'
        est_nodes = geff_estimated_nodes(gpath) if gpath.exists() else None
        shp = meta['shape'] if meta and len(meta.get('shape', [])) == 4 else (None, None, None, None)
        rows.append({
            'name': name, 'embryo': embryo_id(name),
            'shape_T': shp[0], 'shape_Z': shp[1], 'shape_Y': shp[2], 'shape_X': shp[3],
            'dtype': meta['dtype'] if meta else None,
            'chunk_shape': meta['chunk_shape'] if meta else None,
            'has_geff': gpath.exists(),
            'estimated_number_of_nodes': est_nodes,
        })
    return pd.DataFrame(rows)

In [121]:
train_names = list_zarr_names(TRAIN_DIR)
test_names  = list_zarr_names(TEST_DIR)
train_embryos = set(map(embryo_id, train_names)) if train_names else set()
test_embryos  = set(map(embryo_id, test_names)) if test_names else set()

split_summary = pd.DataFrame([
    {'split': 'train', 'n_samples': len(train_names), 'n_embryos': len(train_embryos)},
    {'split': 'test',  'n_samples': len(test_names),  'n_embryos': len(test_embryos)},
])
display(split_summary)
save_table(split_summary, 'phase2', 'split_summary')

if train_embryos and test_embryos:
    overlap = sorted(train_embryos & test_embryos)
    print('Embryo overlap between train/test (should be empty):', overlap if overlap else 'none (embryo-disjoint, as documented)')

train_info = scan_split(TRAIN_DIR, CFG.EDA_SAMPLE_LIMIT) if (CFG.RUN_BASIC_EDA and TRAIN_DIR) else pd.DataFrame()
test_info  = scan_split(TEST_DIR,  CFG.EDA_SAMPLE_LIMIT) if (CFG.RUN_BASIC_EDA and TEST_DIR)  else pd.DataFrame()
if len(train_info): display(train_info)
if len(test_info): display(test_info)
save_table(train_info, 'phase2', 'train_sample_metadata')
save_table(test_info, 'phase2', 'test_sample_metadata')

,split,n_samples,n_embryos
0,train,199,2
1,test,4,2


Embryo overlap between train/test (should be empty): ['44b6', '6bba']


,name,embryo,shape_T,shape_Z,shape_Y,shape_X,dtype,chunk_shape,has_geff,estimated_number_of_nodes
0,44b6_0113de3b,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
1,6bba_05b6850b,6bba,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
2,44b6_0b24845f,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
3,44b6_0c582fdc,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
4,44b6_0db75fae,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
5,44b6_12dfb391,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
6,44b6_144b256d,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None
7,44b6_1574802b,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",True,None


,name,embryo,shape_T,shape_Z,shape_Y,shape_X,dtype,chunk_shape,has_geff,estimated_number_of_nodes
0,44b6_0113de3b,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",False,None
1,6bba_05b6850b,6bba,100,64,256,256,uint16,"[1, 64, 256, 256]",False,None
2,44b6_0b24845f,44b6,100,64,256,256,uint16,"[1, 64, 256, 256]",False,None
3,6bba_05db0fb1,6bba,100,64,256,256,uint16,"[1, 64, 256, 256]",False,None


PosixPath('/kaggle/working/outputs/tables/phase2_test_sample_metadata.csv')

In [122]:
# ---- Figure 01: split / embryo counts ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].bar(split_summary['split'], split_summary['n_samples'], color=[COLORS['blue'], COLORS['orange']])
axes[0].set_title('Samples per split'); axes[0].set_ylabel('count')
axes[1].bar(split_summary['split'], split_summary['n_embryos'], color=[COLORS['green'], COLORS['purple']])
axes[1].set_title('Unique embryos per split'); axes[1].set_ylabel('count')
save_fig(fig, 'phase2', '01_split_embryo_counts')
plt.show()

In [123]:
# ---- Figure 02: array geometry consistency ----
if len(train_info):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
    for ax, col, title in zip(axes, ['shape_T', 'shape_Z', 'shape_Y'], ['T (frames)', 'Z (depth)', 'Y (height)']):
        vals = train_info[col].dropna()
        nb_ = min(10, max(3, vals.nunique())) if len(vals) else 3
        ax.hist(vals, bins=nb_, color=COLORS['teal'], edgecolor='white')
        ax.set_title(title)
    save_fig(fig, 'phase2', '02_array_geometry_hist')
    plt.show()
else:
    print('train_info empty -> skipping geometry histogram (no data mounted).')

In [124]:
# ---- Figure 03: estimated_number_of_nodes distribution ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(train_info) and train_info['estimated_number_of_nodes'].notna().any():
    ax.hist(train_info['estimated_number_of_nodes'].dropna(), bins=12, color=COLORS['blue'],
            alpha=0.8, edgecolor='white', label='train (sampled)')
ax.set_title('estimated_number_of_nodes (sampled)')
ax.set_xlabel('estimated_number_of_nodes'); ax.set_ylabel('count'); ax.legend()
save_fig(fig, 'phase2', '03_estimated_nodes_hist')
plt.show()

In [125]:
# ---- Figure 04: estimated nodes per frame (density proxy) ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(train_info) and train_info['estimated_number_of_nodes'].notna().any():
    density = (train_info['estimated_number_of_nodes'] / train_info['shape_T']).dropna()
    ax.bar(range(len(density)), sorted(density.values), color=COLORS['orange'])
    ax.set_title('Estimated nodes / frame, sorted (sampled train)')
    ax.set_xlabel('sample rank'); ax.set_ylabel('est. nodes per frame')
save_fig(fig, 'phase2', '04_estimated_nodes_per_frame')
plt.show()

## Phase 3 — Ground-Truth Graph EDA

Training labels are sparse, so this is not a full census of cells, but it measures the geometry of
annotated events that drive every linking/division hyperparameter:

| statistic | modelling decision |
|---|---|
| edge displacement (um) | choose / sanity-check `MAX_LINK_DIST_UM` |
| edge time gap dt | decide whether adjacent-frame-only links suffice |
| division count & daughter distance | tune `DETECT_DIVISIONS`, `DIV_PARENT_DIST_UM`, `DIV_SISTER_DIST_UM` |
| labelled vs. estimated node ratio | quantify annotation sparsity |

In [126]:
def load_geff_raw(geff_path):
    if zarr is None or geff_path is None or not Path(geff_path).exists():
        return None, None
    try:
        root = zarr.open_group(str(geff_path), mode='r')
        node_ids = np.asarray(root['nodes/ids'][:]).astype(np.int64)
        data = {'node_id': node_ids}
        for key in ('t', 'z', 'y', 'x'):
            data[key] = np.asarray(root[f'nodes/props/{key}/values'][:])
        nodes = pd.DataFrame(data)
        edges_arr = np.asarray(root['edges/ids'][:]).astype(np.int64)
        edges = (pd.DataFrame(edges_arr, columns=['source_id', 'target_id'])
                 if len(edges_arr) else pd.DataFrame(columns=['source_id', 'target_id']))
        return nodes, edges
    except Exception as exc:
        print('load_geff_raw failed for', geff_path, ':', exc)
        return None, None

def geff_stats_for_sample(name, split_dir=None):
    split_dir = split_dir or TRAIN_DIR
    geff_path = Path(split_dir) / f'{name}.geff'
    nodes, edges = load_geff_raw(geff_path)
    if nodes is None:
        return None
    nodes = nodes.set_index('node_id')
    n_nodes, n_edges = len(nodes), len(edges)
    est = geff_estimated_nodes(geff_path)

    out_deg = edges['source_id'].value_counts() if n_edges else pd.Series(dtype=int)
    n_divisions = int((out_deg >= 2).sum())

    disp_um, dt_vals, sister_dists = [], [], []
    if n_edges:
        src = nodes.reindex(edges['source_id']).reset_index(drop=True)
        tgt = nodes.reindex(edges['target_id']).reset_index(drop=True)
        valid = src[['t', 'z', 'y', 'x']].notna().all(axis=1) & tgt[['t', 'z', 'y', 'x']].notna().all(axis=1)
        src, tgt = src[valid], tgt[valid]
        scale = np.array(CFG.SCALE)
        delta = (tgt[['z', 'y', 'x']].values - src[['z', 'y', 'x']].values) * scale
        disp_um = np.linalg.norm(delta, axis=1)
        dt_vals = (tgt['t'].values - src['t'].values)

        div_sources = out_deg[out_deg >= 2].index
        for s in div_sources:
            children = edges.loc[edges['source_id'] == s, 'target_id']
            pts = nodes.reindex(children)[['z', 'y', 'x']].dropna().values * scale
            if len(pts) >= 2:
                sister_dists.append(float(np.linalg.norm(pts[0] - pts[1])))

    return {
        'name': name, 'embryo': embryo_id(name), 'n_labelled_nodes': n_nodes, 'n_edges': n_edges,
        'estimated_number_of_nodes': est,
        'label_ratio': (n_nodes / est) if est else np.nan,
        'n_divisions': n_divisions,
        'edge_disp_um_median': float(np.median(disp_um)) if len(disp_um) else np.nan,
        'edge_disp_um_p95': float(np.percentile(disp_um, 95)) if len(disp_um) else np.nan,
        'edge_dt_mode': int(pd.Series(dt_vals).mode().iloc[0]) if len(dt_vals) else np.nan,
        'edge_dt_gt1_frac': float(np.mean(np.asarray(dt_vals) > 1)) if len(dt_vals) else np.nan,
        'sister_dist_um_median': float(np.median(sister_dists)) if sister_dists else np.nan,
        '_disp_um': disp_um, '_dt': dt_vals, '_sister': sister_dists,
    }

In [127]:
if not CFG.RUN_GT_EDA:
    print('RUN_GT_EDA=False; GT graph EDA skipped.')
elif not train_names:
    print('No train data found; GT graph EDA skipped.')
else:
    gt_sample_names = select_diverse_samples(train_names, CFG.GT_EDA_SAMPLE_LIMIT)
    display(pd.DataFrame({'gt_sample': gt_sample_names, 'embryo': [embryo_id(n) for n in gt_sample_names]}))

    gt_rows = []
    for name in gt_sample_names:
        row = geff_stats_for_sample(name)
        if row is not None:
            gt_rows.append(row)

    gt_info = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in gt_rows])
    ALL_DISP = np.concatenate([r['_disp_um'] for r in gt_rows if len(r['_disp_um'])]) if gt_rows else np.array([])
    ALL_DT = np.concatenate([np.asarray(r['_dt']) for r in gt_rows if len(r['_dt'])]) if gt_rows else np.array([])
    ALL_SISTER = np.concatenate([r['_sister'] for r in gt_rows if len(r['_sister'])]) if gt_rows else np.array([])

    if len(gt_info):
        display(gt_info)
        numeric_cols = [c for c in gt_info.columns if c not in {'name', 'embryo'}]
        display(gt_info[numeric_cols].describe())
        save_table(gt_info, 'phase3', 'gt_sample_stats')
        save_table(gt_info[numeric_cols].describe().reset_index(), 'phase3', 'gt_sample_stats_describe')
    else:
        print('No readable .geff files found among sampled training names.')

,gt_sample,embryo
0,44b6_0113de3b,44b6
1,6bba_05b6850b,6bba
2,44b6_0b24845f,44b6
3,44b6_0c582fdc,44b6
4,44b6_0db75fae,44b6
5,44b6_12dfb391,44b6


No readable .geff files found among sampled training names.


In [128]:
# ---- Figure 05: edge displacement distribution (physical, um) ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(ALL_DISP):
    ax.hist(ALL_DISP, bins=40, color=COLORS['blue'], edgecolor='white')
    for q, c in zip([50, 90, 95, 99], [COLORS['green'], COLORS['orange'], COLORS['red'], COLORS['purple']]):
        v = np.percentile(ALL_DISP, q)
        ax.axvline(v, color=c, linestyle='--', linewidth=1, label=f'p{q}={v:.1f}um')
    ax.axvline(CFG.MAX_LINK_DIST_UM, color='black', linewidth=1.4, label=f'MAX_LINK_DIST_UM={CFG.MAX_LINK_DIST_UM}')
    ax.legend(fontsize=7)
ax.set_title('GT edge displacement (um)'); ax.set_xlabel('um'); ax.set_ylabel('count')
save_fig(fig, 'phase3', '05_edge_displacement_hist')
plt.show()

In [129]:
# ---- Figure 06: edge time-gap (dt) distribution ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(ALL_DT):
    vals, counts = np.unique(ALL_DT, return_counts=True)
    ax.bar(vals, counts, color=COLORS['teal'])
ax.set_title('GT edge time gap (dt = t_target - t_source)')
ax.set_xlabel('dt (frames)'); ax.set_ylabel('count')
save_fig(fig, 'phase3', '06_edge_dt_hist')
plt.show()

In [130]:
# ---- Figure 07: sister-cell (division daughter) distance ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(ALL_SISTER):
    ax.hist(ALL_SISTER, bins=20, color=COLORS['purple'], edgecolor='white')
    ax.axvline(CFG.DIV_SISTER_DIST_UM, color='black', linestyle='--', label=f'DIV_SISTER_DIST_UM={CFG.DIV_SISTER_DIST_UM}')
    ax.legend()
ax.set_title('Division daughter-daughter distance (um)')
ax.set_xlabel('um'); ax.set_ylabel('count')
save_fig(fig, 'phase3', '07_division_sister_distance')
plt.show()

In [131]:
# ---- Figure 08: labelled-node ratio per sample (annotation sparsity) ----
fig, ax = plt.subplots(figsize=(8, 4))
if 'gt_info' in dir() and len(gt_info):
    order = gt_info.sort_values('label_ratio')
    ax.barh([short for short in order['name']], order['label_ratio'], color=COLORS['orange'])
    ax.set_xlabel('n_labelled_nodes / estimated_number_of_nodes')
    ax.set_title('Annotation sparsity per sample')
save_fig(fig, 'phase3', '08_label_ratio_per_sample')
plt.show()

In [132]:
# ---- Figure 09: divisions per sample + nodes vs edges scatter ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if 'gt_info' in dir() and len(gt_info):
    axes[0].bar(gt_info['name'], gt_info['n_divisions'], color=COLORS['red'])
    axes[0].set_title('Division events per sample'); axes[0].tick_params(axis='x', rotation=60)
    axes[1].scatter(gt_info['n_labelled_nodes'], gt_info['n_edges'], color=COLORS['blue'], s=40)
    axes[1].set_xlabel('n_labelled_nodes'); axes[1].set_ylabel('n_edges')
    axes[1].set_title('Labelled nodes vs. edges')
save_fig(fig, 'phase3', '09_divisions_and_node_edge_scatter')
plt.show()

In [133]:
# ---- Figure 10: per-embryo aggregated GT statistics ----
fig, ax = plt.subplots(figsize=(8, 4))
if 'gt_info' in dir() and len(gt_info):
    by_embryo = gt_info.groupby('embryo')[['n_labelled_nodes', 'n_edges', 'n_divisions']].sum()
    by_embryo.plot(kind='bar', ax=ax, color=[COLORS['blue'], COLORS['green'], COLORS['red']])
    ax.set_title('GT totals aggregated per embryo')
    save_table(by_embryo.reset_index(), 'phase3', 'gt_stats_by_embryo')
save_fig(fig, 'phase3', '10_gt_stats_by_embryo')
plt.show()

## Phase 4 — Classical Baseline Detector

Keeps full Z resolution, performs X/Y block-mean pooling by `XY_DS` to reach a near-isotropic working grid,
smooths, thresholds with `theta = max(theta_Otsu, P50 + alpha*(P99.8-P50))`, finds local maxima at least
`MIN_PEAK_DIST` apart, maps back to original voxel coordinates, and optionally refines each centroid by a
local center-of-mass in the raw volume.

In [134]:
def load_volume(zarr_path, t, shape=None, dtype=np.uint16):
    chunk = Path(zarr_path) / '0' / 'c' / str(t) / '0' / '0' / '0'
    if blosc2 is not None and chunk.exists():
        try:
            raw = blosc2.decompress(chunk.read_bytes())
            arr = np.frombuffer(raw, dtype=dtype)
            if shape is not None:
                return arr.reshape(shape[1:]).copy()
            return arr.copy()
        except Exception:
            pass
    if zarr is not None:
        arr = zarr.open_array(str(Path(zarr_path) / '0'), mode='r')
        return np.asarray(arr[t], dtype=dtype)
    raise ImportError('Either blosc2 or zarr is required to read Zarr volumes.')

def block_mean_xy(vol, factor):
    z, y, x = vol.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    arr = vol[:, :y2, :x2].astype(np.float32)
    arr = arr.reshape(z, y2 // factor, factor, x2 // factor, factor)
    return arr.mean(axis=(2, 4))

def threshold_components(sm, cfg):
    bg = float(np.percentile(sm, 50.0))
    hi = float(np.percentile(sm, cfg.THRESH_HI_PERCENTILE))
    perc_floor = bg + cfg.THRESH_REL * (hi - bg)
    try:
        otsu = float(threshold_otsu(sm)) if threshold_otsu is not None else perc_floor
    except Exception:
        otsu = perc_floor
    final = max(otsu, perc_floor)
    driver = 'otsu' if otsu >= perc_floor else 'percentile'
    return {'threshold_otsu': otsu, 'threshold_percentile': perc_floor,
            'threshold_rel': cfg.THRESH_REL, 'threshold_final': final, 'threshold_driver': driver}

def detect_centroids_classical(vol, cfg=None):
    cfg = cfg or CFG
    ds = block_mean_xy(vol, cfg.XY_DS)
    sm = gaussian_filter(ds, sigma=cfg.SMOOTH_SIGMA)
    comps = threshold_components(sm, cfg)
    theta = comps['threshold_final']

    if peak_local_max is not None:
        coords = peak_local_max(sm, min_distance=cfg.MIN_PEAK_DIST, threshold_abs=theta, exclude_border=False)
    else:
        mx = maximum_filter(sm, size=cfg.MIN_PEAK_DIST)
        mask = (sm == mx) & (sm >= theta)
        coords = np.argwhere(mask)

    if len(coords) == 0:
        return np.zeros((0, 3)), comps

    # map working-grid (z, y_ds, x_ds) back to original voxel coordinates
    centers = coords.astype(np.float32).copy()
    centers[:, 1] = centers[:, 1] * cfg.XY_DS + cfg.XY_DS / 2.0
    centers[:, 2] = centers[:, 2] * cfg.XY_DS + cfg.XY_DS / 2.0

    if cfg.USE_CENTROID_REFINEMENT:
        refined = []
        for z, y, x in centers:
            z0, z1 = max(0, int(z) - cfg.REFINE_RADIUS_Z), min(vol.shape[0], int(z) + cfg.REFINE_RADIUS_Z + 1)
            y0, y1 = max(0, int(y) - cfg.REFINE_RADIUS_YX), min(vol.shape[1], int(y) + cfg.REFINE_RADIUS_YX + 1)
            x0, x1 = max(0, int(x) - cfg.REFINE_RADIUS_YX), min(vol.shape[2], int(x) + cfg.REFINE_RADIUS_YX + 1)
            patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float32)
            if patch.sum() > 0:
                cz, cy, cx = center_of_mass(patch)
                refined.append((z0 + cz, y0 + cy, x0 + cx))
            else:
                refined.append((z, y, x))
        centers = np.array(refined, dtype=np.float32)

    if cfg.USE_PHYSICAL_NMS and len(centers) > 1:
        centers = physical_nms(centers, cfg.NMS_RADIUS_UM, cfg.SCALE)

    if cfg.USE_BORDER_FILTER and len(centers) > 0:
        keep = border_keep_mask(centers, vol.shape, cfg.BORDER_KEEP_QUANTILE)
        centers = centers[keep]

    return centers, comps

def physical_nms(centers, radius_um, scale):
    scale = np.array(scale)
    pts_um = centers * scale
    tree = cKDTree(pts_um)
    pairs = tree.query_pairs(r=radius_um)
    suppressed = set()
    for i, j in pairs:
        if i in suppressed or j in suppressed:
            continue
        suppressed.add(j)
    keep_idx = [i for i in range(len(centers)) if i not in suppressed]
    return centers[keep_idx]

def border_keep_mask(centers, vol_shape, keep_quantile):
    z, y, x = vol_shape
    margin_y, margin_x = int(y * keep_quantile), int(x * keep_quantile)
    in_y = (centers[:, 1] >= margin_y) & (centers[:, 1] < y - margin_y)
    in_x = (centers[:, 2] >= margin_x) & (centers[:, 2] < x - margin_x)
    in_z = (centers[:, 0] >= 0) & (centers[:, 0] < z)
    return in_y & in_x & in_z

In [135]:
def _effective_max_link(cfg):
    return cfg.MAX_LINK_DIST_UM * (0.85 if cfg.TIGHT_LINK_PROFILE else 1.0)

def _effective_div_parent(cfg):
    return cfg.DIV_PARENT_DIST_UM

def _effective_div_sister(cfg):
    return cfg.DIV_SISTER_DIST_UM

effective_tracking_geometry = pd.DataFrame([
    {'quantity': 'link gate (um)', 'value': _effective_max_link(CFG)},
    {'quantity': 'division parent gate (um)', 'value': _effective_div_parent(CFG)},
    {'quantity': 'division sister gate (um)', 'value': _effective_div_sister(CFG)},
    {'quantity': 'division midpoint gate (um)', 'value': CFG.DIV_MIDPOINT_DIST if CFG.DIV_USE_MIDPOINT else np.nan},
    {'quantity': 'tight profile enabled', 'value': bool(CFG.TIGHT_LINK_PROFILE)},
    {'quantity': 'isolated pruning enabled', 'value': bool(CFG.PRUNE_ISOLATED_NODES)},
])
display(effective_tracking_geometry)
save_table(effective_tracking_geometry, 'phase4', 'effective_tracking_geometry')

,quantity,value
0,link gate (um),10.5
1,division parent gate (um),9.0
2,division sister gate (um),9.0
3,division midpoint gate (um),6.0
4,tight profile enabled,False
5,isolated pruning enabled,False


PosixPath('/kaggle/working/outputs/tables/phase4_effective_tracking_geometry.csv')

In [136]:
# Pick one demo sample + frame for the visual-QA figures below.
DEMO_NAME = select_diverse_samples(train_names, 1)[0] if train_names else None
DEMO_T = 10
demo_vol, demo_centers_classical, demo_comps = None, None, None

if DEMO_NAME is not None:
    zpath = Path(TRAIN_DIR) / f'{DEMO_NAME}.zarr'
    meta = read_zarr_meta(zpath)
    shape = meta['shape'] if meta else None
    dtype = np.dtype(meta['dtype']) if meta and meta['dtype'] else np.uint16
    try:
        demo_vol = load_volume(zpath, DEMO_T, shape=shape, dtype=dtype)
        demo_centers_classical, demo_comps = detect_centroids_classical(demo_vol, CFG)
        print(f'{DEMO_NAME} t={DEMO_T}: volume shape={demo_vol.shape}, classical detections={len(demo_centers_classical)}')
        print('threshold components:', demo_comps)
    except Exception as exc:
        print('Could not load demo volume (data likely not mounted):', exc)
else:
    print('No training samples available; classical detector demo skipped.')

44b6_0113de3b t=10: volume shape=(64, 256, 256), classical detections=143
threshold components: {'threshold_otsu': 390.7476806640625, 'threshold_percentile': 375.7800549316406, 'threshold_rel': 0.18, 'threshold_final': 390.7476806640625, 'threshold_driver': 'otsu'}


In [137]:
# ---- Figure 11: XY max-intensity projection with detected centers ----
fig, ax = plt.subplots(figsize=(6, 6))
if demo_vol is not None:
    proj = demo_vol.max(axis=0)
    ax.imshow(proj, cmap='gray')
    if demo_centers_classical is not None and len(demo_centers_classical):
        ax.scatter(demo_centers_classical[:, 2], demo_centers_classical[:, 1], s=14,
                   facecolors='none', edgecolors=COLORS['red'], linewidths=0.8)
    ax.set_title(f'{DEMO_NAME} t={DEMO_T}: XY max-proj + classical centers')
else:
    ax.text(0.5, 0.5, 'no data mounted', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase4', '11_xy_projection_centers')
plt.show()

In [138]:
# ---- Figure 13: middle-Z slice with overlaid centers in that slice band ----
fig, ax = plt.subplots(figsize=(6, 6))
if demo_vol is not None:
    zc = demo_vol.shape[0] // 2
    ax.imshow(demo_vol[zc], cmap='gray')
    if demo_centers_classical is not None and len(demo_centers_classical):
        near = np.abs(demo_centers_classical[:, 0] - zc) <= 1
        pts = demo_centers_classical[near]
        ax.scatter(pts[:, 2], pts[:, 1], s=16, facecolors='none', edgecolors=COLORS['orange'])
    ax.set_title(f'Middle Z slice (z={zc if demo_vol is not None else "?"}) with nearby centers')
else:
    ax.text(0.5, 0.5, 'no data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase4', '13_middle_z_slice')
plt.show()

In [139]:
# ---- Figure 14: smoothed working grid (max-proj) ----
fig, ax = plt.subplots(figsize=(6, 6))
if demo_vol is not None:
    ds = block_mean_xy(demo_vol, CFG.XY_DS)
    sm = gaussian_filter(ds, sigma=CFG.SMOOTH_SIGMA)
    ax.imshow(sm.max(axis=0), cmap='magma')
    ax.set_title('Smoothed working-grid XY max-proj')
else:
    ax.text(0.5, 0.5, 'no data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase4', '14_smoothed_working_grid')
plt.show()

In [140]:
# ---- Figure 15: intensity histogram with thresholds overlaid ----
fig, ax = plt.subplots(figsize=(7, 4))
if demo_vol is not None:
    ds = block_mean_xy(demo_vol, CFG.XY_DS)
    sm = gaussian_filter(ds, sigma=CFG.SMOOTH_SIGMA)
    ax.hist(sm.ravel(), bins=80, color=COLORS['gray'], alpha=0.8)
    ax.axvline(demo_comps['threshold_otsu'], color=COLORS['blue'], linestyle='--', label='otsu')
    ax.axvline(demo_comps['threshold_percentile'], color=COLORS['orange'], linestyle='--', label='percentile floor')
    ax.axvline(demo_comps['threshold_final'], color=COLORS['red'], linewidth=1.5, label='final (max)')
    ax.set_yscale('log'); ax.legend()
    ax.set_title('Working-grid intensity histogram + thresholds')
save_fig(fig, 'phase4', '15_intensity_histogram_thresholds')
plt.show()

In [141]:
# ---- Figure 16: link preview between two consecutive frames (classical detections) ----
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
link_disp_um = np.array([])
if demo_vol is not None and DEMO_NAME is not None:
    try:
        vol2 = load_volume(Path(TRAIN_DIR) / f'{DEMO_NAME}.zarr', DEMO_T + 1, shape=shape, dtype=dtype)
        centers2, _ = detect_centroids_classical(vol2, CFG)
        scale = np.array(CFG.SCALE)
        if len(demo_centers_classical) and len(centers2):
            d = cdist(demo_centers_classical * scale, centers2 * scale)
            row_ind, col_ind = linear_sum_assignment(d)
            gate = CFG.MAX_LINK_DIST_UM
            keep = d[row_ind, col_ind] <= gate
            link_disp_um = d[row_ind, col_ind][keep]

            proj = demo_vol.max(axis=0)
            axes[0].imshow(proj, cmap='gray')
            for r, c, ok in zip(row_ind, col_ind, keep):
                if not ok: continue
                p0, p1 = demo_centers_classical[r], centers2[c]
                axes[0].annotate('', xy=(p1[2], p1[1]), xytext=(p0[2], p0[1]),
                                  arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=0.8))
            axes[0].set_title(f't={DEMO_T} -> t={DEMO_T+1} link preview')

            axes[1].hist(link_disp_um, bins=20, color=COLORS['teal'], edgecolor='white')
            axes[1].axvline(gate, color='black', linestyle='--', label=f'gate={gate}um')
            axes[1].set_title('Frame-pair link displacement (um)'); axes[1].legend()
    except Exception as exc:
        print('link preview skipped:', exc)
save_fig(fig, 'phase4', '16_link_preview_and_displacement')
plt.show()

## Phase 5 — GPU Deep-Learning Detector (heatmap-regression 3D CNN)

A small 3D CNN regresses a per-voxel Gaussian-blob heatmap (one blob per GT centroid) on the same
near-isotropic working grid used by the classical detector. It is **deliberately tiny and time-boxed**
(few channels, few epochs, a capped number of patches) so it trains in a few minutes on a Kaggle T4/P100 and
is skipped automatically without GPU/torch. This is a complement to the classical detector, not a
replacement — Phase 6 compares both.

In [142]:
def make_gaussian_heatmap(shape, centers_vox, sigma):
    '''centers_vox: (N,3) array of (z, y, x) on the *working* grid.'''
    hm = np.zeros(shape, dtype=np.float32)
    if len(centers_vox) == 0:
        return hm
    zz, yy, xx = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]), indexing='ij')
    for c in centers_vox:
        z0, y0, x0 = c
        d2 = (zz - z0) ** 2 + (yy - y0) ** 2 + (xx - x0) ** 2
        hm = np.maximum(hm, np.exp(-d2 / (2 * sigma ** 2)))
    return hm

def build_training_patches(cfg, names, n_samples, patches_per_sample, train_dir):
    '''Extract (volume_patch, heatmap_patch) pairs around GT centroids for DL training.'''
    patches_x, patches_y = [], []
    chosen = select_diverse_samples(names, n_samples)
    for name in chosen:
        zpath = Path(train_dir) / f'{name}.zarr'
        gpath = Path(train_dir) / f'{name}.geff'
        meta = read_zarr_meta(zpath)
        if meta is None or not gpath.exists():
            continue
        shape = meta['shape']
        dtype = np.dtype(meta['dtype']) if meta['dtype'] else np.uint16
        nodes, _ = load_geff_raw(gpath)
        if nodes is None or len(nodes) == 0:
            continue
        by_t = nodes.groupby('t')
        frames_with_nodes = [int(t) for t in by_t.groups.keys()]
        if not frames_with_nodes:
            continue
        np.random.shuffle(frames_with_nodes)
        taken = 0
        for t in frames_with_nodes:
            if taken >= patches_per_sample:
                break
            try:
                vol = load_volume(zpath, t, shape=shape, dtype=dtype)
            except Exception:
                continue
            ds = block_mean_xy(vol, cfg.XY_DS)
            pts = by_t.get_group(t)[['z', 'y', 'x']].values.astype(np.float32)
            pts_ds = pts.copy()
            pts_ds[:, 1] /= cfg.XY_DS
            pts_ds[:, 2] /= cfg.XY_DS

            Z, Y, X = ds.shape
            pz, py, px = min(cfg.DL_PATCH_Z, Z), min(cfg.DL_PATCH_Y, Y), min(cfg.DL_PATCH_X, X)
            z0 = np.random.randint(0, max(1, Z - pz + 1))
            y0 = np.random.randint(0, max(1, Y - py + 1))
            x0 = np.random.randint(0, max(1, X - px + 1))
            crop = ds[z0:z0 + pz, y0:y0 + py, x0:x0 + px]
            local_pts = pts_ds - np.array([z0, y0, x0])
            in_patch = ((local_pts >= 0) & (local_pts < np.array([pz, py, px]))).all(axis=1)
            local_pts = local_pts[in_patch]
            if len(local_pts) == 0:
                continue
            hm = make_gaussian_heatmap(crop.shape, local_pts, cfg.DL_HEATMAP_SIGMA_VOX)
            crop_norm = (crop - crop.mean()) / (crop.std() + 1e-6)
            patches_x.append(crop_norm.astype(np.float32))
            patches_y.append(hm.astype(np.float32))
            taken += 1
    return patches_x, patches_y

In [143]:
DL_TRAINED = False
dl_history = []

if not (TORCH_OK and GPU_AVAILABLE and CFG.USE_DL_DETECTOR):
    print('Skipping DL training: requires torch + CUDA GPU + USE_DL_DETECTOR=True.')
    print(f'  torch ok={TORCH_OK}, gpu available={GPU_AVAILABLE}, USE_DL_DETECTOR={CFG.USE_DL_DETECTOR}')
elif not train_names:
    print('Skipping DL training: no training data mounted.')
else:
    class TinyUNet3D(nn.Module):
        '''Very small 3-level 3D U-Net for heatmap regression. Kept narrow for speed.'''
        def __init__(self, base=8):
            super().__init__()
            def block(cin, cout):
                return nn.Sequential(
                    nn.Conv3d(cin, cout, 3, padding=1), nn.InstanceNorm3d(cout), nn.ReLU(inplace=True),
                    nn.Conv3d(cout, cout, 3, padding=1), nn.InstanceNorm3d(cout), nn.ReLU(inplace=True),
                )
            self.enc1 = block(1, base)
            self.enc2 = block(base, base * 2)
            self.enc3 = block(base * 2, base * 4)
            self.pool = nn.MaxPool3d(2)
            self.up2 = nn.ConvTranspose3d(base * 4, base * 2, 2, stride=2)
            self.dec2 = block(base * 4, base * 2)
            self.up1 = nn.ConvTranspose3d(base * 2, base, 2, stride=2)
            self.dec1 = block(base * 2, base)
            self.out = nn.Conv3d(base, 1, 1)

        def forward(self, x):
            e1 = self.enc1(x)
            e2 = self.enc2(self.pool(e1))
            e3 = self.enc3(self.pool(e2))
            d2 = self.dec2(torch.cat([self.up2(e3), e2], dim=1))
            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
            return self.out(d1)

    t0 = time.time()
    px, py = build_training_patches(CFG, train_names, CFG.DL_TRAIN_SAMPLES, CFG.DL_PATCHES_PER_SAMPLE, TRAIN_DIR)
    print(f'Built {len(px)} training patches in {time.time()-t0:.1f}s')

    if len(px) < 4:
        print('Too few patches extracted (need >=4); skipping DL training.')
    else:
        # pad/crop every patch to a common shape so they can be batched
        pz = min(CFG.DL_PATCH_Z, min(p.shape[0] for p in px))
        py_ = min(CFG.DL_PATCH_Y, min(p.shape[1] for p in px))
        px_ = min(CFG.DL_PATCH_X, min(p.shape[2] for p in px))
        pz, py_, px_ = [max(8, (v // 4) * 4) for v in (pz, py_, px_)]  # divisible by 4 for 2 pool levels

        def fit_shape(arr):
            return arr[:pz, :py_, :px_]

        X = np.stack([fit_shape(a) for a in px])[:, None]
        Y = np.stack([fit_shape(a) for a in py])[:, None]
        X_t = torch.from_numpy(X).float()
        Y_t = torch.from_numpy(Y).float()
        n = X_t.shape[0]
        n_val = max(1, int(0.2 * n))
        perm = torch.randperm(n)
        val_idx, train_idx = perm[:n_val], perm[n_val:]

        model = TinyUNet3D(base=8).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=CFG.DL_LR)
        bce = nn.BCEWithLogitsLoss()

        t0 = time.time()
        for epoch in range(CFG.DL_EPOCHS):
            model.train()
            perm_ep = train_idx[torch.randperm(len(train_idx))]
            losses = []
            for i in range(0, len(perm_ep), CFG.DL_BATCH_SIZE):
                idx = perm_ep[i:i + CFG.DL_BATCH_SIZE]
                xb, yb = X_t[idx].to(DEVICE), Y_t[idx].to(DEVICE)
                opt.zero_grad()
                pred = model(xb)
                loss = bce(pred, yb)
                loss.backward(); opt.step()
                losses.append(loss.item())
            model.eval()
            with torch.no_grad():
                vpred = model(X_t[val_idx].to(DEVICE))
                vloss = bce(vpred, Y_t[val_idx].to(DEVICE)).item()
            dl_history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'val_loss': vloss})
            print(f'epoch {epoch+1}/{CFG.DL_EPOCHS}  train_loss={np.mean(losses):.4f}  val_loss={vloss:.4f}')
            if time.time() - t0 > 600:   # hard 10-minute training budget safeguard
                print('Training time budget reached; stopping early.')
                break

        DL_TRAINED = True
        torch.save(model.state_dict(), MODEL_DIR / 'tiny_unet3d_heatmap.pt')
        print('Saved model to', MODEL_DIR / 'tiny_unet3d_heatmap.pt')

dl_history_df = pd.DataFrame(dl_history)
if len(dl_history_df):
    display(dl_history_df)
save_table(dl_history_df, 'phase5', 'dl_training_history')

Built 0 training patches in 0.0s
Too few patches extracted (need >=4); skipping DL training.


PosixPath('/kaggle/working/outputs/tables/phase5_dl_training_history.csv')

In [144]:
# ---- Figure 17: DL training/validation loss curve ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(dl_history_df):
    ax.plot(dl_history_df['epoch'], dl_history_df['train_loss'], marker='o', color=COLORS['blue'], label='train')
    ax.plot(dl_history_df['epoch'], dl_history_df['val_loss'], marker='o', color=COLORS['red'], label='val')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'DL training skipped (no GPU/data)', ha='center', va='center'); ax.axis('off')
ax.set_title('DL detector: BCE loss per epoch'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
save_fig(fig, 'phase5', '17_dl_loss_curve')
plt.show()

In [145]:
def detect_centroids_dl(vol, model, cfg, device):
    '''Run the trained heatmap CNN on a *whole* working-grid volume via simple tiling, return centroids
    in original-volume voxel coordinates.'''
    ds = block_mean_xy(vol, cfg.XY_DS)
    norm = (ds - ds.mean()) / (ds.std() + 1e-6)
    Z, Y, X = norm.shape
    pad_z = (-Z) % 4; pad_y = (-Y) % 4; pad_x = (-X) % 4
    padded = np.pad(norm, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='reflect')
    with torch.no_grad():
        xb = torch.from_numpy(padded[None, None]).float().to(device)
        logits = model(xb)
        heat = torch.sigmoid(logits)[0, 0].cpu().numpy()
    heat = heat[:Z, :Y, :X]
    if peak_local_max is not None:
        coords = peak_local_max(heat, min_distance=cfg.DL_MIN_PEAK_DIST, threshold_abs=cfg.DL_PEAK_THRESH)
    else:
        mx = maximum_filter(heat, size=cfg.DL_MIN_PEAK_DIST)
        coords = np.argwhere((heat == mx) & (heat >= cfg.DL_PEAK_THRESH))
    if len(coords) == 0:
        return np.zeros((0, 3)), heat
    centers = coords.astype(np.float32)
    centers[:, 1] = centers[:, 1] * cfg.XY_DS + cfg.XY_DS / 2.0
    centers[:, 2] = centers[:, 2] * cfg.XY_DS + cfg.XY_DS / 2.0
    return centers, heat

demo_centers_dl, demo_heat = None, None
if DL_TRAINED and demo_vol is not None:
    try:
        demo_centers_dl, demo_heat = detect_centroids_dl(demo_vol, model, CFG, DEVICE)
        print('DL detections on demo frame:', len(demo_centers_dl))
    except Exception as exc:
        print('DL inference on demo frame failed:', exc)

In [146]:
# ---- Figure 18: DL heatmap overlay + DL vs classical centers side-by-side ----
fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
if demo_vol is not None:
    proj = demo_vol.max(axis=0)
    axes[0].imshow(proj, cmap='gray')
    if demo_centers_classical is not None and len(demo_centers_classical):
        axes[0].scatter(demo_centers_classical[:, 2], demo_centers_classical[:, 1], s=14,
                         facecolors='none', edgecolors=COLORS['red'], label='classical')
    axes[0].set_title('Classical detections'); axes[0].legend(loc='lower right', fontsize=7)

    axes[1].imshow(proj, cmap='gray')
    if demo_centers_dl is not None and len(demo_centers_dl):
        axes[1].scatter(demo_centers_dl[:, 2], demo_centers_dl[:, 1], s=14,
                         facecolors='none', edgecolors=COLORS['teal'], label='DL')
        axes[1].set_title('DL detections')
    else:
        axes[1].set_title('DL detections (unavailable: no GPU training run)')
    axes[1].legend(loc='lower right', fontsize=7)
else:
    for ax in axes: ax.text(0.5, 0.5, 'no data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase5', '18_classical_vs_dl_detections')
plt.show()

## Phase 6 — Detector Comparison, Calibration Sweep & Feature Importance

Three things happen here, all metric-oriented (not just visual QA):

1. A `THRESH_REL x MIN_PEAK_DIST` sweep on embryo-diverse training samples, scored against
   `rho = N_hat / N_est` and sparse one-to-one recall within `MATCH_GATE_UM`.
2. A classical-vs-DL count/recall comparison on the same samples (DL skipped gracefully if untrained).
3. A small tabular **feature-importance study**: a RandomForest classifies "is this peak a true detection"
   from local peak features (intensity, prominence, local density, depth, distance to nearest neighbour);
   importances are read off via permutation importance and, where available, **SHAP**; a simple **LIME**-style
   local explanation is produced for one example. This explains *why* the detector accepts/rejects a peak,
   which is independent of, and complementary to, the geometric calibration sweep above.

In [147]:
def point_match_recall(gt_xyz, pred_xyz, gate_um):
    '''One-to-one (Hungarian) bipartite match within `gate_um`; returns matched fraction of GT points.'''
    if len(gt_xyz) == 0:
        return np.nan, 0
    if len(pred_xyz) == 0:
        return 0.0, 0
    d = cdist(gt_xyz, pred_xyz)
    row_ind, col_ind = linear_sum_assignment(np.where(d <= gate_um, d, 1e6))
    matched = (d[row_ind, col_ind] <= gate_um).sum()
    return matched / len(gt_xyz), int(matched)

def evaluate_detector_on_sample(name, detector_fn, cfg, train_dir, n_frames=3):
    zpath = Path(train_dir) / f'{name}.zarr'
    gpath = Path(train_dir) / f'{name}.geff'
    meta = read_zarr_meta(zpath)
    if meta is None or not gpath.exists():
        return None
    shape, dtype = meta['shape'], np.dtype(meta['dtype']) if meta['dtype'] else np.uint16
    nodes, _ = load_geff_raw(gpath)
    if nodes is None or len(nodes) == 0:
        return None
    frames = sorted(nodes['t'].unique())[:n_frames]
    n_pred_total, n_gt_total, n_matched_total = 0, 0, 0
    for t in frames:
        try:
            vol = load_volume(zpath, int(t), shape=shape, dtype=dtype)
        except Exception:
            continue
        pred, _ = detector_fn(vol)
        gt_xyz = nodes.loc[nodes['t'] == t, ['z', 'y', 'x']].values * np.array(cfg.SCALE)
        pred_xyz = pred * np.array(cfg.SCALE) if len(pred) else pred
        recall, matched = point_match_recall(gt_xyz, pred_xyz, cfg.MATCH_GATE_UM)
        n_pred_total += len(pred); n_gt_total += len(gt_xyz); n_matched_total += matched
    est = geff_estimated_nodes(gpath)
    return {
        'name': name, 'n_frames_eval': len(frames), 'n_pred': n_pred_total, 'n_gt_labelled': n_gt_total,
        'n_matched': n_matched_total,
        'sparse_recall': n_matched_total / n_gt_total if n_gt_total else np.nan,
        'rho_vs_estimate': (n_pred_total / len(frames)) / (est / shape[0]) if est else np.nan,
    }

In [148]:
calib_rows = []
sweep_samples = select_diverse_samples(train_names, min(4, CFG.GT_EDA_SAMPLE_LIMIT)) if train_names else []
sweep_grid = [(rel, mpd) for rel in [0.10, 0.18, 0.28] for mpd in [2, 3, 4]]

if sweep_samples:
    for rel, mpd in sweep_grid:
        cfg_try = CONFIG(**{**asdict(CFG), 'THRESH_REL': rel, 'MIN_PEAK_DIST': mpd})
        det_fn = lambda v, c=cfg_try: detect_centroids_classical(v, c)
        for name in sweep_samples:
            res = evaluate_detector_on_sample(name, det_fn, cfg_try, TRAIN_DIR, n_frames=2)
            if res is not None:
                res.update({'THRESH_REL': rel, 'MIN_PEAK_DIST': mpd})
                calib_rows.append(res)

calib_df = pd.DataFrame(calib_rows)
if len(calib_df):
    calib_summary = calib_df.groupby(['THRESH_REL', 'MIN_PEAK_DIST'])[['sparse_recall', 'rho_vs_estimate']].mean().reset_index()
    display(calib_summary)
    save_table(calib_df, 'phase6', 'calibration_sweep_raw')
    save_table(calib_summary, 'phase6', 'calibration_sweep_summary')
else:
    print('Calibration sweep skipped: no training data mounted.')
    calib_summary = pd.DataFrame()

Calibration sweep skipped: no training data mounted.


In [149]:
# ---- Figure 19: calibration sweep - recall heatmap ----
fig, ax = plt.subplots(figsize=(6.5, 5))
if len(calib_summary):
    pivot = calib_summary.pivot(index='THRESH_REL', columns='MIN_PEAK_DIST', values='sparse_recall')
    im = ax.imshow(pivot.values, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    ax.set_xlabel('MIN_PEAK_DIST'); ax.set_ylabel('THRESH_REL')
    plt.colorbar(im, ax=ax, label='sparse recall')
ax.set_title('Calibration sweep: sparse recall')
save_fig(fig, 'phase6', '19_calibration_recall_heatmap')
plt.show()

In [150]:
# ---- Figure 20: calibration sweep - rho (count ratio) heatmap ----
fig, ax = plt.subplots(figsize=(6.5, 5))
if len(calib_summary):
    pivot = calib_summary.pivot(index='THRESH_REL', columns='MIN_PEAK_DIST', values='rho_vs_estimate')
    im = ax.imshow(pivot.values, cmap='coolwarm', aspect='auto', vmin=0, vmax=2)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    ax.set_xlabel('MIN_PEAK_DIST'); ax.set_ylabel('THRESH_REL')
    plt.colorbar(im, ax=ax, label='rho = N_hat / N_est')
ax.set_title('Calibration sweep: count ratio rho')
save_fig(fig, 'phase6', '20_calibration_rho_heatmap')
plt.show()

In [151]:
# ---- Figure 21: recall vs rho trade-off scatter, colored by THRESH_REL ----
fig, ax = plt.subplots(figsize=(7, 5))
if len(calib_summary):
    sc = ax.scatter(calib_summary['rho_vs_estimate'], calib_summary['sparse_recall'],
                     c=calib_summary['THRESH_REL'], cmap='plasma', s=80, edgecolor='k')
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1)
    plt.colorbar(sc, ax=ax, label='THRESH_REL')
ax.set_xlabel('rho = N_hat / N_est'); ax.set_ylabel('sparse recall')
ax.set_title('Recall vs. count-ratio trade-off')
save_fig(fig, 'phase6', '21_recall_vs_rho_tradeoff')
plt.show()

In [152]:
# ---- Classical vs DL comparison table ----
compare_rows = []
if sweep_samples:
    det_classical = lambda v: detect_centroids_classical(v, CFG)
    for name in sweep_samples:
        r = evaluate_detector_on_sample(name, det_classical, CFG, TRAIN_DIR, n_frames=2)
        if r is not None:
            r['detector'] = 'classical'; compare_rows.append(r)
    if DL_TRAINED:
        det_dl = lambda v: detect_centroids_dl(v, model, CFG, DEVICE)
        for name in sweep_samples:
            r = evaluate_detector_on_sample(name, det_dl, CFG, TRAIN_DIR, n_frames=2)
            if r is not None:
                r['detector'] = 'dl'; compare_rows.append(r)

compare_df = pd.DataFrame(compare_rows)
if len(compare_df):
    display(compare_df)
    save_table(compare_df, 'phase6', 'classical_vs_dl_comparison')
else:
    print('Classical-vs-DL comparison skipped (no data, or DL untrained -> classical-only rows above if any).')

Classical-vs-DL comparison skipped (no data, or DL untrained -> classical-only rows above if any).


In [153]:
# ---- Figure 22: classical vs DL recall / rho bar comparison ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if len(compare_df):
    agg = compare_df.groupby('detector')[['sparse_recall', 'rho_vs_estimate']].mean()
    agg['sparse_recall'].plot(kind='bar', ax=axes[0], color=COLORS['blue']); axes[0].set_title('Mean sparse recall')
    agg['rho_vs_estimate'].plot(kind='bar', ax=axes[1], color=COLORS['orange']); axes[1].axhline(1.0, color='gray', linestyle='--')
    axes[1].set_title('Mean rho (count ratio)')
else:
    for ax in axes: ax.text(0.5, 0.5, 'comparison unavailable', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase6', '22_classical_vs_dl_bars')
plt.show()

In [154]:
# ======================================================================
# Tabular feature-importance study (RandomForest + permutation + SHAP/LIME)
# ======================================================================
def build_peak_feature_table(name, cfg, train_dir, n_frames=2):
    '''For one sample: build a table of candidate peaks (pre-NMS/border filter) with local features,
    labelled 1 if within MATCH_GATE_UM of a GT node, else 0. Used purely for an *explanatory* model of
    what makes the detector accept a peak -- not part of the detection pipeline itself.'''
    zpath = Path(train_dir) / f'{name}.zarr'
    gpath = Path(train_dir) / f'{name}.geff'
    meta = read_zarr_meta(zpath)
    if meta is None or not gpath.exists():
        return pd.DataFrame()
    shape, dtype = meta['shape'], np.dtype(meta['dtype']) if meta['dtype'] else np.uint16
    nodes, _ = load_geff_raw(gpath)
    if nodes is None or len(nodes) == 0:
        return pd.DataFrame()
    frames = sorted(nodes['t'].unique())[:n_frames]
    rows = []
    for t in frames:
        try:
            vol = load_volume(zpath, int(t), shape=shape, dtype=dtype)
        except Exception:
            continue
        ds = block_mean_xy(vol, cfg.XY_DS)
        sm = gaussian_filter(ds, sigma=cfg.SMOOTH_SIGMA)
        comps = threshold_components(sm, cfg)
        if peak_local_max is not None:
            coords = peak_local_max(sm, min_distance=1, threshold_abs=comps['threshold_percentile'] * 0.6)
        else:
            mx = maximum_filter(sm, size=2)
            coords = np.argwhere((sm == mx) & (sm >= comps['threshold_percentile'] * 0.6))
        if len(coords) == 0:
            continue
        gt_xyz = nodes.loc[nodes['t'] == t, ['z', 'y', 'x']].values.astype(np.float32)
        gt_xyz_ds = gt_xyz.copy(); gt_xyz_ds[:, 1] /= cfg.XY_DS; gt_xyz_ds[:, 2] /= cfg.XY_DS
        tree = cKDTree(gt_xyz_ds * np.array(cfg.SCALE)) if len(gt_xyz_ds) else None
        peak_tree = cKDTree(coords) if len(coords) > 1 else None
        for c in coords:
            z, y, x = c
            intensity = float(sm[z, y, x])
            local_win = sm[max(0, z-2):z+3, max(0, y-3):y+4, max(0, x-3):x+4]
            prominence = intensity - float(local_win.mean())
            nn_dist_peaks = (peak_tree.query(c, k=2)[0][1] if peak_tree is not None else np.nan)
            depth_frac = z / sm.shape[0]
            border_dist = min(y, sm.shape[1] - y, x, sm.shape[2] - x)
            is_true = 0
            if tree is not None:
                dmin, _ = tree.query((c * np.array(cfg.SCALE)).reshape(1, -1))
                is_true = int(dmin[0] <= cfg.MATCH_GATE_UM)
            rows.append({'intensity': intensity, 'prominence': prominence, 'nn_dist_peaks': nn_dist_peaks,
                         'depth_frac': depth_frac, 'border_dist': border_dist, 'is_true': is_true})
    return pd.DataFrame(rows)

feature_rows = []
if sweep_samples:
    for name in sweep_samples[:3]:
        feature_rows.append(build_peak_feature_table(name, CFG, TRAIN_DIR))
feature_df = pd.concat(feature_rows, ignore_index=True) if feature_rows else pd.DataFrame()
feature_df = feature_df.dropna() if len(feature_df) else feature_df
if len(feature_df):
    display(feature_df.describe())
    save_table(feature_df, 'phase6', 'peak_feature_table')
else:
    print('Peak feature table empty (no data mounted) -> feature-importance study skipped below.')

Peak feature table empty (no data mounted) -> feature-importance study skipped below.


In [155]:
# ---- Figure 23: class balance + feature distributions by label ----
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
if len(feature_df) and feature_df['is_true'].nunique() > 1:
    feature_df['is_true'].value_counts().plot(kind='bar', ax=axes[0], color=[COLORS['gray'], COLORS['green']])
    axes[0].set_title('Peak label balance (0=false,1=true)')
    for ax, col in zip(axes[1:], ['intensity', 'prominence']):
        for lbl, c in zip([0, 1], [COLORS['red'], COLORS['green']]):
            vals = feature_df.loc[feature_df['is_true'] == lbl, col]
            ax.hist(vals, bins=20, alpha=0.6, color=c, label=f'is_true={lbl}')
        ax.set_title(col); ax.legend(fontsize=7)
else:
    for ax in axes: ax.text(0.5, 0.5, 'insufficient data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase6', '23_peak_feature_distributions')
plt.show()

In [156]:
RF_OK = False
rf_model, X_cols = None, ['intensity', 'prominence', 'nn_dist_peaks', 'depth_frac', 'border_dist']
perm_importance_df = pd.DataFrame()

if len(feature_df) >= 30 and feature_df['is_true'].nunique() > 1:
    try:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.inspection import permutation_importance
        from sklearn.model_selection import train_test_split

        Xf = feature_df[X_cols].values
        yf = feature_df['is_true'].values
        Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=0, stratify=yf if len(set(yf)) > 1 else None)
        rf_model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=0, n_jobs=-1)
        rf_model.fit(Xtr, ytr)
        acc = rf_model.score(Xte, yte)
        print(f'RandomForest peak-acceptance classifier accuracy: {acc:.3f}')

        perm = permutation_importance(rf_model, Xte, yte, n_repeats=20, random_state=0)
        perm_importance_df = pd.DataFrame({'feature': X_cols, 'importance_mean': perm.importances_mean,
                                            'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
        display(perm_importance_df)
        save_table(perm_importance_df, 'phase6', 'permutation_feature_importance')
        RF_OK = True
    except Exception as exc:
        print('RandomForest / permutation-importance study failed:', exc)
else:
    print('Not enough labelled peaks for a feature-importance model -> skipped (needs real data + >=2 classes).')

Not enough labelled peaks for a feature-importance model -> skipped (needs real data + >=2 classes).


In [157]:
# ---- Figure 24: permutation feature importance ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(perm_importance_df):
    ax.barh(perm_importance_df['feature'], perm_importance_df['importance_mean'],
            xerr=perm_importance_df['importance_std'], color=COLORS['purple'])
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, 'unavailable', ha='center', va='center'); ax.axis('off')
ax.set_title('Permutation feature importance (peak-acceptance classifier)')
save_fig(fig, 'phase6', '24_permutation_feature_importance')
plt.show()

In [158]:
# ---- SHAP (if installed) ----
shap_summary_df = pd.DataFrame()
if RF_OK:
    try:
        import shap
        explainer = shap.TreeExplainer(rf_model)
        sample_X = feature_df[X_cols].sample(min(200, len(feature_df)), random_state=0)
        shap_values = explainer.shap_values(sample_X)
        sv = shap_values[1] if isinstance(shap_values, list) else shap_values
        mean_abs_shap = np.abs(sv).mean(axis=0)
        shap_summary_df = pd.DataFrame({'feature': X_cols, 'mean_abs_shap': mean_abs_shap}).sort_values('mean_abs_shap', ascending=False)
        display(shap_summary_df)
        save_table(shap_summary_df, 'phase6', 'shap_feature_importance')

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.barh(shap_summary_df['feature'], shap_summary_df['mean_abs_shap'], color=COLORS['teal'])
        ax.invert_yaxis(); ax.set_title('Mean |SHAP value| per feature')
        save_fig(fig, 'phase6', '25_shap_feature_importance')
        plt.show()
    except Exception as exc:
        print('shap unavailable or failed (this is optional and non-fatal):', exc)
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.text(0.5, 0.5, f'SHAP unavailable: {exc}', ha='center', va='center', wrap=True); ax.axis('off')
        save_fig(fig, 'phase6', '25_shap_feature_importance')
        plt.show()
else:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.text(0.5, 0.5, 'SHAP skipped: classifier not trained', ha='center', va='center'); ax.axis('off')
    save_fig(fig, 'phase6', '25_shap_feature_importance')
    plt.show()

In [159]:
# ---- LIME-style local explanation for one example (manual fallback if `lime` not installed) ----
lime_local_df = pd.DataFrame()
if RF_OK:
    try:
        try:
            from lime.lime_tabular import LimeTabularExplainer
            explainer = LimeTabularExplainer(feature_df[X_cols].values, feature_names=X_cols,
                                              class_names=['false', 'true'], discretize_continuous=True)
            idx = int(feature_df.index[feature_df['is_true'] == 1][0]) if (feature_df['is_true'] == 1).any() else 0
            exp = explainer.explain_instance(feature_df.loc[idx, X_cols].values, rf_model.predict_proba, num_features=5)
            lime_local_df = pd.DataFrame(exp.as_list(), columns=['rule', 'weight'])
        except ImportError:
            # Manual local-surrogate fallback: perturb the chosen example and fit a local linear model.
            from sklearn.linear_model import LinearRegression
            idx = int(feature_df.index[feature_df['is_true'] == 1][0]) if (feature_df['is_true'] == 1).any() else 0
            x0 = feature_df.loc[idx, X_cols].values.astype(float)
            rng = np.random.RandomState(0)
            perturbed = x0 + rng.normal(scale=feature_df[X_cols].std().values * 0.25, size=(300, len(X_cols)))
            probs = rf_model.predict_proba(perturbed)[:, 1]
            weights = np.exp(-((perturbed - x0) ** 2).sum(axis=1) / (2 * (feature_df[X_cols].std().values.mean()) ** 2))
            lr = LinearRegression().fit(perturbed, probs, sample_weight=weights)
            lime_local_df = pd.DataFrame({'feature': X_cols, 'local_coefficient': lr.coef_}).sort_values(
                'local_coefficient', key=np.abs, ascending=False)
            print('lime package not installed -> used a manual local-surrogate (LIME-style) explanation instead.')
        display(lime_local_df)
        save_table(lime_local_df, 'phase6', 'lime_local_explanation')
    except Exception as exc:
        print('LIME-style local explanation failed (non-fatal):', exc)
else:
    print('LIME-style local explanation skipped: classifier not trained.')

LIME-style local explanation skipped: classifier not trained.


## Phase 7 — Tracking / Linking

Frame-to-frame linking is a gated linear-sum-assignment problem on physical (um) distance. Division
candidates are found as a second pass: an unmatched detection at `t+1` is linked back to its nearest
existing track within `DIV_PARENT_DIST_UM` if that creates a sister pair within `DIV_SISTER_DIST_UM`
(optionally re-checked against the sister midpoint).

In [160]:
def link_frames(centers_prev, centers_next, cfg):
    '''Hungarian-gated linking between two consecutive frames' centroids (voxel coords).
    Returns list of (i, j) index pairs (prev_idx -> next_idx) that pass the physical gate.'''
    if len(centers_prev) == 0 or len(centers_next) == 0:
        return []
    scale = np.array(cfg.SCALE)
    d = cdist(centers_prev * scale, centers_next * scale)
    gate = _effective_max_link(cfg)
    cost = np.where(d <= gate, d, 1e6)
    row_ind, col_ind = linear_sum_assignment(cost)
    return [(int(r), int(c)) for r, c in zip(row_ind, col_ind) if d[r, c] <= gate]

def detect_divisions(track_state, centers_next, used_next_idx, cfg):
    '''Second pass: try to attach currently-unmatched next-frame detections as a second daughter of an
    existing track if it stays within division gates.'''
    new_edges = []
    if not cfg.DETECT_DIVISIONS or len(centers_next) == 0:
        return new_edges
    scale = np.array(cfg.SCALE)
    unmatched = [j for j in range(len(centers_next)) if j not in used_next_idx]
    if not unmatched:
        return new_edges
    track_positions = {tid: state['pos'] for tid, state in track_state.items()}
    if not track_positions:
        return new_edges
    tids = list(track_positions.keys())
    pos_arr = np.array([track_positions[tid] for tid in tids]) * scale
    for j in unmatched:
        cand = (centers_next[j] * scale).reshape(1, -1)
        dists = cdist(cand, pos_arr)[0]
        best = int(np.argmin(dists))
        if dists[best] <= cfg.DIV_PARENT_DIST_UM:
            parent_tid = tids[best]
            sister_pos = track_state[parent_tid].get('last_daughter_pos')
            if sister_pos is not None:
                sister_dist = float(np.linalg.norm((centers_next[j] - sister_pos) * scale))
                if sister_dist > cfg.DIV_SISTER_DIST_UM:
                    continue
            new_edges.append((parent_tid, j))
            track_state[parent_tid]['last_daughter_pos'] = centers_next[j]
    return new_edges

In [161]:
def run_tracking_on_volume_sequence(zpath, shape, dtype, t_list, detector_fn, cfg):
    '''Streaming per-dataset tracker: detect each frame, link to the previous frame, optionally detect
    divisions. Returns node rows and edge rows (local integer node ids, 0-based per call).'''
    node_rows, edge_rows = [], []
    track_state = {}   # track_id -> {'pos':..., 'node_idx':..., 'last_daughter_pos':...}
    next_track_id = 0
    next_node_id = 0
    prev_centers, prev_node_ids = None, None

    for t in t_list:
        try:
            vol = load_volume(zpath, int(t), shape=shape, dtype=dtype)
        except Exception as exc:
            print(f'  frame {t} failed to load ({exc}); skipping frame.')
            continue
        centers, _ = detector_fn(vol)
        cur_node_ids = []
        for c in centers:
            nid = next_node_id; next_node_id += 1
            node_rows.append({'node_id': nid, 't': int(t), 'z': float(c[0]), 'y': float(c[1]), 'x': float(c[2])})
            cur_node_ids.append(nid)
        cur_node_ids = np.array(cur_node_ids)

        used_next_idx = set()
        if prev_centers is not None and len(centers) > 0:
            pairs = link_frames(prev_centers, centers, cfg)
            for pi, ni in pairs:
                prev_tid = None
                for tid, st in track_state.items():
                    if st['node_idx'] == prev_node_ids[pi]:
                        prev_tid = tid; break
                if prev_tid is None:
                    prev_tid = next_track_id; next_track_id += 1
                edge_rows.append({'source_id': int(prev_node_ids[pi]), 'target_id': int(cur_node_ids[ni])})
                track_state[prev_tid] = {'pos': centers[ni], 'node_idx': cur_node_ids[ni], 'last_daughter_pos': None}
                used_next_idx.add(ni)

            div_edges = detect_divisions(track_state, centers, used_next_idx, cfg)
            for parent_tid, j in div_edges:
                parent_node = track_state[parent_tid]['node_idx']
                edge_rows.append({'source_id': int(parent_node), 'target_id': int(cur_node_ids[j])})

        prev_centers, prev_node_ids = centers, cur_node_ids

    return pd.DataFrame(node_rows), pd.DataFrame(edge_rows)

In [162]:
# Demo: run the full per-sample tracker over a short window for visual QA.
demo_track_nodes, demo_track_edges = pd.DataFrame(), pd.DataFrame()
if DEMO_NAME is not None and TRAIN_DIR is not None:
    try:
        det_fn = (lambda v: detect_centroids_dl(v, model, CFG, DEVICE)) if DL_TRAINED else (lambda v: detect_centroids_classical(v, CFG))
        t_window = list(range(DEMO_T, min(DEMO_T + 6, shape[0])))
        demo_track_nodes, demo_track_edges = run_tracking_on_volume_sequence(
            Path(TRAIN_DIR) / f'{DEMO_NAME}.zarr', shape, dtype, t_window, det_fn, CFG)
        print(f'Demo tracking window {t_window}: {len(demo_track_nodes)} nodes, {len(demo_track_edges)} edges')
        save_table(demo_track_nodes, 'phase7', 'demo_track_nodes')
        save_table(demo_track_edges, 'phase7', 'demo_track_edges')
    except Exception as exc:
        print('Demo tracking run failed:', exc)

Demo tracking window [10, 11, 12, 13, 14, 15]: 824 nodes, 647 edges


In [163]:
# ---- Figure 27: track count over time (nodes per frame, demo window) ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(demo_track_nodes):
    counts = demo_track_nodes.groupby('t').size()
    ax.plot(counts.index, counts.values, marker='o', color=COLORS['blue'])
else:
    ax.text(0.5, 0.5, 'no demo tracking output', ha='center', va='center'); ax.axis('off')
ax.set_title('Detections per frame (demo tracking window)')
ax.set_xlabel('t'); ax.set_ylabel('n detections')
save_fig(fig, 'phase7', '27_track_counts_over_time')
plt.show()

In [164]:
# ---- Figure 28: edge displacement distribution from the demo tracker run ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(demo_track_edges) and len(demo_track_nodes):
    nmap = demo_track_nodes.set_index('node_id')
    src = nmap.reindex(demo_track_edges['source_id'])[['z', 'y', 'x']].values
    tgt = nmap.reindex(demo_track_edges['target_id'])[['z', 'y', 'x']].values
    scale = np.array(CFG.SCALE)
    d = np.linalg.norm((tgt - src) * scale, axis=1)
    ax.hist(d, bins=20, color=COLORS['teal'], edgecolor='white')
    ax.axvline(_effective_max_link(CFG), color='black', linestyle='--', label='link gate')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'no demo tracking output', ha='center', va='center'); ax.axis('off')
ax.set_title('Predicted edge displacement (demo window)')
save_fig(fig, 'phase7', '28_predicted_edge_displacement')
plt.show()

In [165]:
# ---- Figure 29: 3D-ish quiver of predicted links projected on XY ----
fig, ax = plt.subplots(figsize=(6.5, 6.5))
if demo_vol is not None and len(demo_track_edges) and len(demo_track_nodes):
    proj = demo_vol.max(axis=0)
    ax.imshow(proj, cmap='gray', alpha=0.6)
    nmap = demo_track_nodes.set_index('node_id')
    first_t = demo_track_nodes['t'].min()
    relevant = demo_track_edges[demo_track_edges['source_id'].isin(nmap[nmap['t'] == first_t].index)]
    for _, row in relevant.iterrows():
        p0 = nmap.loc[row['source_id'], ['y', 'x']].values
        p1 = nmap.loc[row['target_id'], ['y', 'x']].values
        ax.annotate('', xy=(p1[1], p1[0]), xytext=(p0[1], p0[0]),
                    arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=0.9))
    ax.set_title('First-frame links of the demo tracker (XY projection)')
else:
    ax.text(0.5, 0.5, 'no demo tracking output', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase7', '29_demo_track_links_xy')
plt.show()

## Phase 8 — Local Graph Proxy Scoring

Training labels are sparse, so this is **not** the full official metric, but it answers the two practical
questions before submission: how many nodes does the detector emit relative to `estimated_number_of_nodes`,
and how often do sparse GT nodes get a one-to-one predicted match within `MATCH_GATE_UM`? It also reports a
coarse edge-Jaccard-style proxy and a division-count comparison.

In [166]:
def local_proxy_score_sample(name, detector_fn, cfg, train_dir, n_frames=4):
    zpath = Path(train_dir) / f'{name}.zarr'
    gpath = Path(train_dir) / f'{name}.geff'
    meta = read_zarr_meta(zpath)
    if meta is None or not gpath.exists():
        return None
    shape, dtype = meta['shape'], np.dtype(meta['dtype']) if meta['dtype'] else np.uint16
    nodes, edges = load_geff_raw(gpath)
    if nodes is None or len(nodes) == 0:
        return None

    frames = sorted(nodes['t'].unique())[:n_frames]
    t_list = list(range(int(min(frames)), int(max(frames)) + 1))
    pred_nodes, pred_edges = run_tracking_on_volume_sequence(zpath, shape, dtype, t_list, detector_fn, cfg)
    if len(pred_nodes) == 0:
        return {'name': name, 'sparse_recall': 0.0, 'rho_vs_estimate': np.nan, 'edge_proxy_jaccard': 0.0,
                'n_pred_divisions': 0, 'n_gt_divisions': int((edges['source_id'].value_counts() >= 2).sum()) if len(edges) else 0}

    scale = np.array(cfg.SCALE)
    matched, total_gt = 0, 0
    pred_by_t = {t: pred_nodes[pred_nodes['t'] == t][['z', 'y', 'x']].values for t in frames}
    gt_id_to_pred_id = {}
    for t in frames:
        gt_t = nodes[nodes['t'] == t]
        gt_xyz = gt_t[['z', 'y', 'x']].values * scale
        pred_xyz = pred_by_t.get(t, np.zeros((0, 3))) * scale
        total_gt += len(gt_xyz)
        if len(gt_xyz) == 0 or len(pred_xyz) == 0:
            continue
        d = cdist(gt_xyz, pred_xyz)
        row_ind, col_ind = linear_sum_assignment(np.where(d <= cfg.MATCH_GATE_UM, d, 1e6))
        pred_ids_t = pred_nodes[pred_nodes['t'] == t]['node_id'].values
        for r, c in zip(row_ind, col_ind):
            if d[r, c] <= cfg.MATCH_GATE_UM:
                matched += 1
                gt_id_to_pred_id[int(gt_t.iloc[r]['node_id'])] = int(pred_ids_t[c])

    edge_hits, edge_total = 0, 0
    if len(edges):
        for _, e in edges.iterrows():
            s, tgt = int(e['source_id']), int(e['target_id'])
            if s in gt_id_to_pred_id and tgt in gt_id_to_pred_id:
                edge_total += 1
                ps, pt = gt_id_to_pred_id[s], gt_id_to_pred_id[tgt]
                if ((pred_edges['source_id'] == ps) & (pred_edges['target_id'] == pt)).any():
                    edge_hits += 1
    edge_proxy = edge_hits / edge_total if edge_total else np.nan

    est = geff_estimated_nodes(gpath)
    pred_density = len(pred_nodes) / len(t_list)
    est_density = (est / shape[0]) if est else np.nan
    n_pred_div = int((pred_edges['source_id'].value_counts() >= 2).sum()) if len(pred_edges) else 0
    n_gt_div = int((edges['source_id'].value_counts() >= 2).sum()) if len(edges) else 0

    return {'name': name, 'sparse_recall': matched / total_gt if total_gt else np.nan,
            'rho_vs_estimate': pred_density / est_density if est_density else np.nan,
            'edge_proxy_jaccard': edge_proxy, 'n_pred_divisions': n_pred_div, 'n_gt_divisions': n_gt_div}

In [167]:
local_proxy_rows = []
if CFG.RUN_LOCAL_PROXY and train_names:
    proxy_samples = select_diverse_samples(train_names, CFG.LOCAL_PROXY_SAMPLE_LIMIT)
    det_fn = (lambda v: detect_centroids_dl(v, model, CFG, DEVICE)) if DL_TRAINED else (lambda v: detect_centroids_classical(v, CFG))
    for name in proxy_samples:
        try:
            r = local_proxy_score_sample(name, det_fn, CFG, TRAIN_DIR, n_frames=4)
            if r is not None:
                local_proxy_rows.append(r)
        except Exception as exc:
            print(f'local proxy scoring failed for {name}:', exc)

local_proxy_df = pd.DataFrame(local_proxy_rows)
if len(local_proxy_df):
    display(local_proxy_df)
    display(local_proxy_df.describe())
    save_table(local_proxy_df, 'phase8', 'local_proxy_scores')
else:
    print('RUN_LOCAL_PROXY=False, or no training data mounted -> local proxy scoring skipped.')

RUN_LOCAL_PROXY=False, or no training data mounted -> local proxy scoring skipped.


In [168]:
# ---- Figure 30: local proxy recall / rho per sample ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if len(local_proxy_df):
    axes[0].bar(local_proxy_df['name'], local_proxy_df['sparse_recall'], color=COLORS['blue'])
    axes[0].set_title('Sparse recall per sample'); axes[0].tick_params(axis='x', rotation=60)
    axes[1].bar(local_proxy_df['name'], local_proxy_df['rho_vs_estimate'], color=COLORS['orange'])
    axes[1].axhline(1.0, color='gray', linestyle='--')
    axes[1].set_title('rho per sample'); axes[1].tick_params(axis='x', rotation=60)
else:
    for ax in axes: ax.text(0.5, 0.5, 'unavailable', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase8', '30_local_proxy_recall_rho')
plt.show()

In [169]:
# ---- Figure 31: edge proxy Jaccard + division count comparison ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if len(local_proxy_df):
    axes[0].bar(local_proxy_df['name'], local_proxy_df['edge_proxy_jaccard'], color=COLORS['teal'])
    axes[0].set_title('Edge proxy match rate'); axes[0].tick_params(axis='x', rotation=60)
    width = 0.35
    xs = np.arange(len(local_proxy_df))
    axes[1].bar(xs - width/2, local_proxy_df['n_pred_divisions'], width, label='pred', color=COLORS['blue'])
    axes[1].bar(xs + width/2, local_proxy_df['n_gt_divisions'], width, label='gt', color=COLORS['red'])
    axes[1].set_xticks(xs); axes[1].set_xticklabels(local_proxy_df['name'], rotation=60)
    axes[1].set_title('Predicted vs. GT division counts'); axes[1].legend()
else:
    for ax in axes: ax.text(0.5, 0.5, 'unavailable', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase8', '31_edge_proxy_and_divisions')
plt.show()

In [170]:
def write_experiment_log(df, cfg, out_path=None):
    out_path = out_path or (LOG_DIR / 'experiment_log.csv')
    row = {'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'), 'detector': 'dl' if DL_TRAINED else 'classical'}
    row.update({k: v for k, v in asdict(cfg).items() if not isinstance(v, (tuple, list))})
    if len(df):
        row.update({f'mean_{c}': df[c].mean() for c in df.select_dtypes('number').columns})
    log_df = pd.DataFrame([row])
    if out_path.exists():
        log_df = pd.concat([pd.read_csv(out_path), log_df], ignore_index=True)
    log_df.to_csv(out_path, index=False)
    return log_df

if len(local_proxy_df):
    exp_log = write_experiment_log(local_proxy_df, CFG)
    display(exp_log.tail())

## Phase 9 — Submission Build & Audit

The final submission is a single CSV with columns:
`id, dataset, row_type, node_id, t, z, y, x, source_id, target_id`
Unused fields per row_type get sentinel value `-1`.

In [171]:
SUBMISSION_CSV_PATH = SUB_DIR / 'submission.csv'
SUBMISSION_HEADER = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']

def stream_submission_for_sample(writer_fn, dataset_name, node_rows_df, edge_rows_df, running_id_ref):
    for _, row in node_rows_df.iterrows():
        rid = running_id_ref[0]; running_id_ref[0] += 1
        writer_fn({'id': rid, 'dataset': dataset_name, 'row_type': 'node',
                   'node_id': int(row['node_id']), 't': int(row['t']),
                   'z': round(float(row['z']), 3), 'y': round(float(row['y']), 3), 'x': round(float(row['x']), 3),
                   'source_id': -1, 'target_id': -1})
    for _, row in edge_rows_df.iterrows():
        rid = running_id_ref[0]; running_id_ref[0] += 1
        writer_fn({'id': rid, 'dataset': dataset_name, 'row_type': 'edge',
                   'node_id': -1, 't': -1, 'z': -1.0, 'y': -1.0, 'x': -1.0,
                   'source_id': int(row['source_id']), 'target_id': int(row['target_id'])})

In [172]:
import csv

det_fn_submit = (lambda v: detect_centroids_dl(v, model, CFG, DEVICE)) if DL_TRAINED else (lambda v: detect_centroids_classical(v, CFG))
test_names_submit = list_zarr_names(TEST_DIR) if TEST_DIR else []
limit = CFG.SUBMISSION_LIMIT
names_to_submit = test_names_submit[:limit] if limit else test_names_submit

sub_stats = []
running_id = [0]

print(f'Building submission for {len(names_to_submit)} test samples...')
t_sub_start = time.time()

with open(SUBMISSION_CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=SUBMISSION_HEADER)
    writer.writeheader()

    def write_row(d):
        writer.writerow(d)

    for i, name in enumerate(names_to_submit):
        zpath = Path(TEST_DIR) / f'{name}.zarr'
        meta = read_zarr_meta(zpath)
        if meta is None:
            print(f'  [{i+1}/{len(names_to_submit)}] {name}: zarr.json not found, skipping.')
            continue
        shape = meta['shape']
        dtype = np.dtype(meta['dtype']) if meta['dtype'] else np.uint16
        if len(shape) != 4:
            print(f'  [{i+1}/{len(names_to_submit)}] {name}: unexpected shape {shape}, skipping.')
            continue
        T = shape[0]
        t_list = list(range(T))

        t0 = time.time()
        sample_nodes, sample_edges = run_tracking_on_volume_sequence(
            zpath, shape, dtype, t_list, det_fn_submit, CFG)

        stream_submission_for_sample(write_row, name, sample_nodes, sample_edges, running_id)
        elapsed = time.time() - t0
        sub_stats.append({'name': name, 'T': T, 'n_nodes': len(sample_nodes),
                          'n_edges': len(sample_edges), 'elapsed_s': round(elapsed, 1)})
        print(f'  [{i+1}/{len(names_to_submit)}] {name}: {len(sample_nodes)} nodes, '
              f'{len(sample_edges)} edges, {elapsed:.1f}s')

sub_stats_df = pd.DataFrame(sub_stats)
total_rows = running_id[0]
print(f'\nSubmission complete: {total_rows} rows, {time.time()-t_sub_start:.1f}s total.')
print(f'Saved to: {SUBMISSION_CSV_PATH}')
save_table(sub_stats_df, 'phase9', 'submission_per_sample_stats')
if len(sub_stats_df): display(sub_stats_df)

Building submission for 4 test samples...
  [1/4] 44b6_0113de3b: 14768 nodes, 14260 edges, 17.4s
  [2/4] 44b6_0b24845f: 8076 nodes, 7426 edges, 19.1s
  [3/4] 6bba_05b6850b: 3647 nodes, 3503 edges, 13.5s
  [4/4] 6bba_05db0fb1: 17529 nodes, 16845 edges, 21.9s

Submission complete: 86054 rows, 71.9s total.
Saved to: /kaggle/working/outputs/submission/submission.csv


,name,T,n_nodes,n_edges,elapsed_s
0,44b6_0113de3b,100,14768,14260,17.4
1,44b6_0b24845f,100,8076,7426,19.1
2,6bba_05b6850b,100,3647,3503,13.5
3,6bba_05db0fb1,100,17529,16845,21.9


In [173]:
# ---- Submission schema / value audit ----
audit_rows = []
if SUBMISSION_CSV_PATH.exists():
    sub_preview = pd.read_csv(SUBMISSION_CSV_PATH, nrows=100000)
    if CFG.RUN_SUBMISSION_DISPLAY:
        display(sub_preview.head(20))

    # Schema checks
    missing_cols = [c for c in SUBMISSION_HEADER if c not in sub_preview.columns]
    audit_rows.append({'check': 'missing_columns', 'result': missing_cols if missing_cols else 'none', 'pass': not bool(missing_cols)})

    # Row-type breakdown
    if 'row_type' in sub_preview.columns:
        rt_counts = sub_preview['row_type'].value_counts().to_dict()
        audit_rows.append({'check': 'row_type_values', 'result': str(rt_counts), 'pass': set(rt_counts.keys()) <= {'node', 'edge'}})

    # Node rows: t,z,y,x should be >= 0
    node_rows_sub = sub_preview[sub_preview.get('row_type', '') == 'node'] if 'row_type' in sub_preview.columns else pd.DataFrame()
    if len(node_rows_sub):
        neg_coords = ((node_rows_sub[['t', 'z', 'y', 'x']] < 0).any(axis=1)).sum()
        audit_rows.append({'check': 'node_coords_non_negative', 'result': f'{neg_coords} violations', 'pass': neg_coords == 0})

    # Edge rows: source_id, target_id should be >= 0
    edge_rows_sub = sub_preview[sub_preview.get('row_type', '') == 'edge'] if 'row_type' in sub_preview.columns else pd.DataFrame()
    if len(edge_rows_sub):
        neg_ids = ((edge_rows_sub[['source_id', 'target_id']] < 0).any(axis=1)).sum()
        audit_rows.append({'check': 'edge_ids_non_negative', 'result': f'{neg_ids} violations', 'pass': neg_ids == 0})

    # id monotonicity
    if 'id' in sub_preview.columns:
        monotone = (sub_preview['id'].diff().dropna() >= 0).all()
        audit_rows.append({'check': 'id_monotone', 'result': str(monotone), 'pass': bool(monotone)})

    # Dataset coverage
    if 'dataset' in sub_preview.columns:
        covered = sub_preview['dataset'].nunique()
        audit_rows.append({'check': 'datasets_in_submission', 'result': str(covered), 'pass': True})

audit_df = pd.DataFrame(audit_rows)
if len(audit_df):
    display(audit_df)
    save_table(audit_df, 'phase9', 'submission_audit')
    all_pass = audit_df['pass'].all()
    print('Submission audit:', 'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED (see above)')

,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,0,0,40.893,6.269,173.469,-1,-1
1,1,44b6_0113de3b,node,1,0,47.910,101.845,170.287,-1,-1
2,2,44b6_0113de3b,node,2,0,35.005,93.767,130.002,-1,-1
3,3,44b6_0113de3b,node,3,0,51.098,137.761,174.378,-1,-1
4,4,44b6_0113de3b,node,4,0,61.540,46.066,229.437,-1,-1
5,5,44b6_0113de3b,node,5,0,54.024,57.882,197.993,-1,-1
6,6,44b6_0113de3b,node,6,0,19.914,178.393,82.242,-1,-1
7,7,44b6_0113de3b,node,7,0,0.891,62.353,58.089,-1,-1
8,8,44b6_0113de3b,node,8,0,12.949,141.629,73.564,-1,-1
9,9,44b6_0113de3b,node,9,0,7.909,13.379,82.283,-1,-1


,check,result,pass
0,missing_columns,none,True
1,row_type_values,"{'node': 44020, 'edge': 42034}",True
2,node_coords_non_negative,0 violations,True
3,edge_ids_non_negative,0 violations,True
4,id_monotone,True,True
5,datasets_in_submission,4,True


Submission audit: ALL CHECKS PASSED


In [174]:
# ---- Figure 32: nodes and edges per sample in submission ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if len(sub_stats_df):
    n = min(30, len(sub_stats_df))
    df_ = sub_stats_df.head(n)
    axes[0].bar(range(n), df_['n_nodes'], color=COLORS['blue'])
    axes[0].set_title('Submitted nodes per sample (first 30)')
    axes[0].set_xlabel('sample'); axes[0].set_ylabel('n_nodes')
    axes[1].bar(range(n), df_['n_edges'], color=COLORS['green'])
    axes[1].set_title('Submitted edges per sample (first 30)')
    axes[1].set_xlabel('sample'); axes[1].set_ylabel('n_edges')
else:
    for ax in axes: ax.text(0.5, 0.5, 'no submission data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase9', '32_submission_nodes_edges_per_sample')
plt.show()

In [175]:
# ---- Figure 33: per-sample processing time ----
fig, ax = plt.subplots(figsize=(8, 4))
if len(sub_stats_df):
    n = min(30, len(sub_stats_df))
    df_ = sub_stats_df.head(n)
    ax.bar(range(n), df_['elapsed_s'], color=COLORS['purple'])
    ax.set_title('Per-sample tracking time (s)')
    ax.set_xlabel('sample'); ax.set_ylabel('seconds')
else:
    ax.text(0.5, 0.5, 'no submission data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase9', '33_per_sample_processing_time')
plt.show()

In [176]:
# ---- Figure 34: submission rows by type (pie) ----
fig, ax = plt.subplots(figsize=(5, 5))
if SUBMISSION_CSV_PATH.exists():
    sub_chunk = pd.read_csv(SUBMISSION_CSV_PATH, nrows=500000)
    if 'row_type' in sub_chunk.columns:
        vc = sub_chunk['row_type'].value_counts()
        ax.pie(vc.values, labels=vc.index, autopct='%1.1f%%',
               colors=[COLORS['blue'], COLORS['orange']])
        ax.set_title('Submission row-type breakdown')
    else:
        ax.text(0.5, 0.5, 'row_type column missing', ha='center', va='center'); ax.axis('off')
else:
    ax.text(0.5, 0.5, 'no submission', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase9', '34_submission_row_type_pie')
plt.show()

In [177]:
# ---- Figure 35: node depth distribution in submission (z-histogram) ----
fig, ax = plt.subplots(figsize=(7, 4))
if SUBMISSION_CSV_PATH.exists():
    sub_chunk = pd.read_csv(SUBMISSION_CSV_PATH, nrows=500000)
    node_z = sub_chunk[sub_chunk.get('row_type', '') == 'node']['z'].dropna() if 'row_type' in sub_chunk.columns else pd.Series()
    if len(node_z):
        ax.hist(node_z.values, bins=40, color=COLORS['teal'], edgecolor='white')
        ax.set_xlabel('z (voxels)'); ax.set_ylabel('count')
        ax.set_title('z-depth distribution of submitted nodes')
    else:
        ax.text(0.5, 0.5, 'no node rows', ha='center', va='center'); ax.axis('off')
else:
    ax.text(0.5, 0.5, 'no submission', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase9', '35_submission_node_z_distribution')
plt.show()

## Phase 10 — Model Performance Comparison, Recommendations & Hyperparameter Tuning Analysis

This phase synthesizes everything learned in previous phases into:
1. A ranked **model-comparison table** (classical vs DL, across sampled embryos)
2. A **hyperparameter sensitivity ranking** derived from the calibration sweep (Phase 6)
3. A simulated **Optuna-style tuning history** visualization (bayesian vs grid comparison)
4. Concrete **improvement recommendations** per failure mode
5. A **radar chart** comparing detector profiles across 5 key axes

In [178]:
# ---- 10.1  Unified model comparison table ----
comparison_records = []

# Pull from local_proxy_df (already computed per-sample scores for the active detector)
if len(local_proxy_df):
    mean_scores = local_proxy_df[['sparse_recall', 'rho_vs_estimate', 'edge_proxy_jaccard',
                                   'n_pred_divisions', 'n_gt_divisions']].mean()
    comparison_records.append({
        'model': f'{"DL-CNN" if DL_TRAINED else "Classical"} (active)',
        'sparse_recall_mean': round(mean_scores['sparse_recall'], 4),
        'rho_mean': round(mean_scores['rho_vs_estimate'], 4),
        'edge_jaccard_mean': round(mean_scores['edge_proxy_jaccard'], 4),
        'division_recall': round(mean_scores['n_pred_divisions'] / max(mean_scores['n_gt_divisions'], 1), 4),
        'notes': 'current active pipeline'
    })

# Pull classical scores separately (always available from calib sweep)
if len(calib_df):
    best_calib = calib_summary.sort_values('sparse_recall', ascending=False).iloc[0]
    comparison_records.append({
        'model': f"Classical (best sweep: α={best_calib['THRESH_REL']}, mpd={best_calib['MIN_PEAK_DIST']})",
        'sparse_recall_mean': round(best_calib['sparse_recall'], 4),
        'rho_mean': round(best_calib['rho_vs_estimate'], 4),
        'edge_jaccard_mean': np.nan,
        'division_recall': np.nan,
        'notes': 'best grid-search params'
    })
    worst_calib = calib_summary.sort_values('sparse_recall').iloc[0]
    comparison_records.append({
        'model': f"Classical (worst sweep: α={worst_calib['THRESH_REL']}, mpd={worst_calib['MIN_PEAK_DIST']})",
        'sparse_recall_mean': round(worst_calib['sparse_recall'], 4),
        'rho_mean': round(worst_calib['rho_vs_estimate'], 4),
        'edge_jaccard_mean': np.nan,
        'division_recall': np.nan,
        'notes': 'worst grid-search params (lower bound)'
    })

# DL trained comparison (if run)
if DL_TRAINED and len(compare_df):
    dl_rows = compare_df[compare_df['detector'] == 'dl']
    cl_rows = compare_df[compare_df['detector'] == 'classical']
    if len(dl_rows):
        comparison_records.append({
            'model': 'DL-CNN TinyUNet3D',
            'sparse_recall_mean': round(dl_rows['sparse_recall'].mean(), 4),
            'rho_mean': round(dl_rows['rho_vs_estimate'].mean(), 4),
            'edge_jaccard_mean': np.nan,
            'division_recall': np.nan,
            'notes': f'trained {CFG.DL_EPOCHS} epochs, {len(dl_history_df)} loss points'
        })
    if len(cl_rows):
        comparison_records.append({
            'model': 'Classical (default CFG)',
            'sparse_recall_mean': round(cl_rows['sparse_recall'].mean(), 4),
            'rho_mean': round(cl_rows['rho_vs_estimate'].mean(), 4),
            'edge_jaccard_mean': np.nan,
            'division_recall': np.nan,
            'notes': 'default CFG params'
        })

if not comparison_records:
    comparison_records.append({
        'model': 'Classical (CFG defaults)', 'sparse_recall_mean': np.nan,
        'rho_mean': np.nan, 'edge_jaccard_mean': np.nan, 'division_recall': np.nan,
        'notes': 'run with real data to populate'
    })

model_comparison_df = pd.DataFrame(comparison_records)
display(model_comparison_df)
save_table(model_comparison_df, 'phase10', 'model_comparison_table')

,model,sparse_recall_mean,rho_mean,edge_jaccard_mean,division_recall,notes
0,Classical (CFG defaults),NaN,NaN,NaN,NaN,run with real data to populate


PosixPath('/kaggle/working/outputs/tables/phase10_model_comparison_table.csv')

In [179]:
# ---- Figure 36: model comparison horizontal bar chart ----
fig, axes = plt.subplots(1, 2, figsize=(13, max(3, len(model_comparison_df) * 0.9 + 1)), constrained_layout=True)
mc = model_comparison_df.copy().fillna(0)
for ax, col, title, color in zip(
        axes,
        ['sparse_recall_mean', 'rho_mean'],
        ['Sparse Recall (higher=better)', 'Rho = N_hat/N_est (1.0 is ideal)'],
        [COLORS['blue'], COLORS['orange']]):
    ax.barh(mc['model'], mc[col], color=color, edgecolor='white')
    ax.set_xlabel(col); ax.set_title(title)
    if col == 'rho_mean':
        ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='ideal=1.0')
        ax.legend(fontsize=7)
save_fig(fig, 'phase10', '36_model_comparison_bars')
plt.show()

In [180]:
# ---- 10.2  Hyperparameter sensitivity ranking from calibration sweep ----
hp_sensitivity = pd.DataFrame()
if len(calib_df) and calib_df['sparse_recall'].notna().any():
    # Variance of recall per hyperparameter dimension
    by_rel = calib_df.groupby('THRESH_REL')['sparse_recall'].mean()
    by_mpd = calib_df.groupby('MIN_PEAK_DIST')['sparse_recall'].mean()
    hp_sensitivity = pd.DataFrame([
        {'hyperparameter': 'THRESH_REL', 'recall_range': round(by_rel.max() - by_rel.min(), 4),
         'recall_std': round(by_rel.std(), 4), 'best_value': float(by_rel.idxmax()),
         'recommendation': f'use α={by_rel.idxmax():.2f}'},
        {'hyperparameter': 'MIN_PEAK_DIST', 'recall_range': round(by_mpd.max() - by_mpd.min(), 4),
         'recall_std': round(by_mpd.std(), 4), 'best_value': float(by_mpd.idxmax()),
         'recommendation': f'use mpd={int(by_mpd.idxmax())}'},
    ]).sort_values('recall_range', ascending=False)
    display(hp_sensitivity)
    save_table(hp_sensitivity, 'phase10', 'hyperparameter_sensitivity')
else:
    print('Hyperparameter sensitivity table: requires calibration sweep data (real data needed).')

Hyperparameter sensitivity table: requires calibration sweep data (real data needed).


In [181]:
# ---- Figure 37: hyperparameter sensitivity (recall range per HP) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
if len(calib_df) and calib_df['sparse_recall'].notna().any():
    by_rel = calib_df.groupby('THRESH_REL')['sparse_recall'].mean()
    by_mpd = calib_df.groupby('MIN_PEAK_DIST')['sparse_recall'].mean()
    axes[0].plot(by_rel.index, by_rel.values, marker='o', color=COLORS['blue']); axes[0].set_title('Recall vs THRESH_REL')
    axes[0].set_xlabel('THRESH_REL'); axes[0].set_ylabel('mean sparse recall')
    axes[1].plot(by_mpd.index.astype(int), by_mpd.values, marker='s', color=COLORS['green']); axes[1].set_title('Recall vs MIN_PEAK_DIST')
    axes[1].set_xlabel('MIN_PEAK_DIST'); axes[1].set_ylabel('mean sparse recall')
else:
    for ax in axes: ax.text(0.5, 0.5, 'no sweep data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase10', '37_hp_sensitivity_lines')
plt.show()

In [182]:
# ---- Figure 38: simulated Bayesian vs Grid-Search tuning history ----
rng = np.random.RandomState(42)
n_trials = 30

# Grid: fixed deterministic coverage
grid_evals = [(a, b) for a in [0.10, 0.18, 0.28] for b in [2, 3, 4]]
# Simulate recall scores based on observed calib if available, else synthetic
def recall_from_params(rel, mpd, calib_df):
    if len(calib_df) and calib_df['sparse_recall'].notna().any():
        row = calib_df[(calib_df['THRESH_REL'] == rel) & (calib_df['MIN_PEAK_DIST'] == mpd)]
        if len(row):
            return float(row['sparse_recall'].mean()) + rng.normal(0, 0.01)
    # Synthetic: concave surface peaked around (0.18, 3)
    return 0.5 - 0.8 * (rel - 0.18) ** 2 - 0.05 * (mpd - 3) ** 2 + rng.normal(0, 0.015)

grid_history = [recall_from_params(r, m, calib_df if 'calib_df' in dir() else pd.DataFrame()) for r, m in grid_evals]
grid_best = [max(grid_history[:i+1]) for i in range(len(grid_history))]

# Bayesian: simulated improvement trajectory (sub-linear convergence)
bayes_history = sorted([recall_from_params(
    rng.choice([0.10, 0.13, 0.18, 0.22, 0.28, 0.33]),
    rng.choice([2, 3, 4, 5]),
    pd.DataFrame()) for _ in range(n_trials)])
bayes_best = np.maximum.accumulate(bayes_history[::-1])[::-1]
bayes_best = np.maximum.accumulate([recall_from_params(
    0.10 + rng.beta(2, 2) * 0.26,
    int(2 + rng.beta(2, 2) * 3),
    pd.DataFrame()) for _ in range(n_trials)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(grid_best)+1), grid_best, marker='s', color=COLORS['orange'], label='Grid Search (exhaustive)')
ax.plot(range(1, len(bayes_best)+1), bayes_best, marker='o', color=COLORS['blue'], label='Bayesian Opt (simulated)')
ax.set_xlabel('# evaluations'); ax.set_ylabel('best sparse recall so far')
ax.set_title('Bayesian vs. Grid Search tuning convergence (simulated)')
ax.legend()
save_fig(fig, 'phase10', '38_hp_tuning_convergence')
plt.show()

In [183]:
# ---- Figure 39: Radar chart comparing detector profiles ----
import matplotlib.patches as mpatches

categories = ['Recall', 'Count Accuracy', 'Division F1', 'Speed', 'Robustness']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

# Scores: [recall, count_accuracy, division_f1, speed, robustness] normalised 0-1
def get_profile(model_name, mc_df):
    row = mc_df[mc_df['model'].str.startswith(model_name)]
    if len(row):
        rec = float(row.iloc[0]['sparse_recall_mean'])
        rho = float(row.iloc[0]['rho_mean'])
        count_acc = max(0, 1 - abs(rho - 1.0)) if not np.isnan(rho) else 0.5
        div = float(row.iloc[0]['division_recall']) if not np.isnan(row.iloc[0]['division_recall']) else 0.5
        return [max(0, min(1, rec)) if not np.isnan(rec) else 0.5, count_acc, max(0, min(1, div)), 0.75, 0.70]
    return [0.5, 0.5, 0.5, 0.75, 0.70]

classical_profile = get_profile('Classical', model_comparison_df) if len(model_comparison_df) else [0.60, 0.55, 0.40, 0.90, 0.75]
dl_profile       = get_profile('DL',       model_comparison_df) if len(model_comparison_df) else [0.70, 0.65, 0.55, 0.50, 0.65]
# Hypothetical enhanced profile (aspirational)
enhanced_profile = [min(1, classical_profile[i] * 1.15) for i in range(N)]

profiles = [
    ('Classical Detector', classical_profile, COLORS['blue']),
    ('DL TinyUNet3D',      dl_profile,        COLORS['green']),
    ('Enhanced (target)',   enhanced_profile,  COLORS['red']),
]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for label, vals, color in profiles:
    v = vals + vals[:1]
    ax.plot(angles, v, color=color, linewidth=2, label=label)
    ax.fill(angles, v, color=color, alpha=0.12)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1); ax.set_title('Detector profile radar chart', pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=8)
save_fig(fig, 'phase10', '39_radar_detector_profiles')
plt.show()

In [184]:
# ---- Figure 40: comprehensive analysis summary grid ----
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
axes = axes.ravel()

# 40a: node density across submission samples
ax = axes[0]
if len(sub_stats_df) and 'n_nodes' in sub_stats_df.columns and 'T' in sub_stats_df.columns:
    dens = sub_stats_df['n_nodes'] / sub_stats_df['T']
    ax.hist(dens, bins=15, color=COLORS['blue'], edgecolor='white')
    ax.set_title('Predicted node density (nodes/frame)'); ax.set_xlabel('nodes/frame')
else:
    ax.text(0.5, 0.5, 'no submission data', ha='center', va='center'); ax.axis('off')

# 40b: edge / node ratio
ax = axes[1]
if len(sub_stats_df) and 'n_edges' in sub_stats_df.columns and 'n_nodes' in sub_stats_df.columns:
    ratio = sub_stats_df['n_edges'] / (sub_stats_df['n_nodes'] + 1)
    ax.bar(range(len(ratio)), sorted(ratio.values), color=COLORS['green'])
    ax.set_title('Edge / node ratio per sample'); ax.set_xlabel('sample rank')
else:
    ax.text(0.5, 0.5, 'no submission data', ha='center', va='center'); ax.axis('off')

# 40c: GT edge displacement vs link gate sensitivity
ax = axes[2]
if len(ALL_DISP):
    gates = np.linspace(2, 20, 80)
    recall_at_gate = [np.mean(ALL_DISP <= g) for g in gates]
    ax.plot(gates, recall_at_gate, color=COLORS['purple'])
    ax.axvline(CFG.MAX_LINK_DIST_UM, color='red', linestyle='--', label=f'current gate={CFG.MAX_LINK_DIST_UM}')
    ax.set_title('GT edge recall vs. link gate (um)'); ax.set_xlabel('gate (um)'); ax.set_ylabel('fraction captured')
    ax.legend(fontsize=7)
else:
    ax.text(0.5, 0.5, 'no GT displacement data', ha='center', va='center'); ax.axis('off')

# 40d: calibration sweep recall vs rho scatter (all grid points)
ax = axes[3]
if len(calib_summary):
    sc = ax.scatter(calib_summary['rho_vs_estimate'], calib_summary['sparse_recall'],
                     c=calib_summary['THRESH_REL'], cmap='plasma', s=70, edgecolor='k')
    ax.axvline(1.0, color='gray', linestyle='--')
    plt.colorbar(sc, ax=ax, label='THRESH_REL')
    ax.set_xlabel('rho'); ax.set_ylabel('recall'); ax.set_title('Recall vs Rho (sweep)')
else:
    ax.text(0.5, 0.5, 'no sweep data', ha='center', va='center'); ax.axis('off')

# 40e: DL loss curve (if trained)
ax = axes[4]
if len(dl_history_df):
    ax.plot(dl_history_df['epoch'], dl_history_df['train_loss'], marker='o', color=COLORS['blue'], label='train')
    ax.plot(dl_history_df['epoch'], dl_history_df['val_loss'], marker='o', color=COLORS['red'], label='val')
    ax.legend(fontsize=7); ax.set_title('DL training loss')
else:
    ax.text(0.5, 0.5, 'DL not trained', ha='center', va='center'); ax.axis('off')

# 40f: node z-distribution heatmap-style (xy vs z density of submitted nodes)
ax = axes[5]
if SUBMISSION_CSV_PATH.exists():
    sc_ = pd.read_csv(SUBMISSION_CSV_PATH, nrows=50000)
    nsc = sc_[sc_.get('row_type', '') == 'node'] if 'row_type' in sc_.columns else pd.DataFrame()
    if len(nsc) and 'z' in nsc.columns and 'y' in nsc.columns:
        ax.hexbin(nsc['y'].values, nsc['z'].values, gridsize=30, cmap='YlOrRd', mincnt=1)
        ax.set_xlabel('y (voxels)'); ax.set_ylabel('z (voxels)')
        ax.set_title('Submitted node density: Y vs Z')
    else:
        ax.text(0.5, 0.5, 'no node rows', ha='center', va='center'); ax.axis('off')
else:
    ax.text(0.5, 0.5, 'no submission', ha='center', va='center'); ax.axis('off')

save_fig(fig, 'phase10', '40_comprehensive_analysis_grid')
plt.show()

In [185]:
# ---- 10.3  Failure mode analysis and concrete recommendations ----
failure_mode_df = pd.DataFrame([
    {'failure_mode': 'Under-detection (rho < 0.7)',
     'symptom': 'sparse_recall low, rho much less than 1',
     'root_cause': 'THRESH_REL too high or SMOOTH_SIGMA over-smoothing',
     'fix': 'Reduce THRESH_REL from 0.18 to 0.10; reduce SMOOTH_SIGMA z-component'},
    {'failure_mode': 'Over-detection (rho > 1.5)',
     'symptom': 'rho >> 1, low precision (false positives near border)',
     'root_cause': 'THRESH_REL too low, NMS radius too small',
     'fix': 'Increase THRESH_REL to 0.28; increase NMS_RADIUS_UM; enable USE_BORDER_FILTER'},
    {'failure_mode': 'Missed divisions',
     'symptom': 'n_pred_divisions << n_gt_divisions',
     'root_cause': 'DIV_PARENT_DIST_UM or DIV_SISTER_DIST_UM too tight',
     'fix': 'Increase DIV_PARENT_DIST_UM to 10; increase DIV_SISTER_DIST_UM to 10; set DIV_USE_MIDPOINT=False'},
    {'failure_mode': 'False divisions',
     'symptom': 'n_pred_divisions >> n_gt_divisions',
     'root_cause': 'DIV gates too loose; dense regions create spurious matches',
     'fix': 'Tighten DIV_SISTER_DIST_UM to 6; enable DIV_USE_MIDPOINT with DIV_MIDPOINT_DIST=5'},
    {'failure_mode': 'Link breaks at large displacement',
     'symptom': 'edge_proxy_jaccard low; cells fast-moving',
     'root_cause': 'MAX_LINK_DIST_UM too small relative to cell motion',
     'fix': 'Increase MAX_LINK_DIST_UM to 15 um; consult edge displacement p95 from Phase 3'},
    {'failure_mode': 'DL detector mis-calibrated threshold',
     'symptom': 'DL recall drops vs classical despite lower loss',
     'root_cause': 'DL_PEAK_THRESH set too high on heatmap sigmoid output',
     'fix': 'Sweep DL_PEAK_THRESH in [0.25, 0.35, 0.45, 0.55]; pick via recall on val frames'},
    {'failure_mode': 'Embryo-specific detection failure',
     'symptom': 'Wide rho variance across samples from same embryo',
     'root_cause': 'Fluorescence intensity varies across embryos/FOVs',
     'fix': 'Enable per-sample adaptive threshold (set THRESH_REL per-sample based on p99-p50 gap)'},
])
display(failure_mode_df)
save_table(failure_mode_df, 'phase10', 'failure_mode_analysis')

,failure_mode,symptom,root_cause,fix
0,Under-detection (rho < 0.7),"sparse_recall low, rho much less than 1",THRESH_REL too high or SMOOTH_SIGMA over-smoot...,Reduce THRESH_REL from 0.18 to 0.10; reduce SM...
1,Over-detection (rho > 1.5),"rho >> 1, low precision (false positives near ...","THRESH_REL too low, NMS radius too small",Increase THRESH_REL to 0.28; increase NMS_RADI...
2,Missed divisions,n_pred_divisions << n_gt_divisions,DIV_PARENT_DIST_UM or DIV_SISTER_DIST_UM too t...,Increase DIV_PARENT_DIST_UM to 10; increase DI...
3,False divisions,n_pred_divisions >> n_gt_divisions,DIV gates too loose; dense regions create spur...,Tighten DIV_SISTER_DIST_UM to 6; enable DIV_US...
4,Link breaks at large displacement,edge_proxy_jaccard low; cells fast-moving,MAX_LINK_DIST_UM too small relative to cell mo...,Increase MAX_LINK_DIST_UM to 15 um; consult ed...
5,DL detector mis-calibrated threshold,DL recall drops vs classical despite lower loss,DL_PEAK_THRESH set too high on heatmap sigmoid...,"Sweep DL_PEAK_THRESH in [0.25, 0.35, 0.45, 0.5..."
6,Embryo-specific detection failure,Wide rho variance across samples from same embryo,Fluorescence intensity varies across embryos/FOVs,Enable per-sample adaptive threshold (set THRE...


PosixPath('/kaggle/working/outputs/tables/phase10_failure_mode_analysis.csv')

In [186]:
# ---- Figure 41: failure mode priority matrix (impact vs ease of fix) ----
# Subjective scores for visualization
impact = [0.9, 0.85, 0.7, 0.6, 0.75, 0.65, 0.80]
ease   = [0.85, 0.80, 0.60, 0.60, 0.70, 0.75, 0.50]
labels = failure_mode_df['failure_mode'].tolist()

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(ease, impact, s=140, c=np.arange(len(labels)), cmap='tab10', edgecolor='k', zorder=3)
for i, lbl in enumerate(labels):
    ax.annotate(f'{i+1}', (ease[i], impact[i]), ha='center', va='center', fontsize=7, fontweight='bold', color='white', zorder=4)
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.7, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Ease of fix (1=easiest)'); ax.set_ylabel('Impact on recall (1=highest)')
ax.set_title('Failure mode priority matrix')
legend_patches = [mpatches.Patch(color=scatter.cmap(scatter.norm(i)), label=f'{i+1}: {l[:40]}') for i, l in enumerate(labels)]
ax.legend(handles=legend_patches, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=6.5)
ax.set_xlim(0.3, 1.05); ax.set_ylim(0.4, 1.05)
save_fig(fig, 'phase10', '41_failure_mode_priority_matrix')
plt.show()

In [187]:
# ---- 10.4  Enhanced pipeline recommendations table ----
enhancement_df = pd.DataFrame([
    {'priority': 1, 'enhancement': 'Larger DL model (UNet++ or 3D ResUNet)',
     'expected_gain': '+5-15% recall', 'effort': 'medium',
     'implementation': 'Replace TinyUNet3D with 16-channel UNet++; train 15-20 epochs; requires 8GB+ VRAM'},
    {'priority': 2, 'enhancement': 'Per-sample adaptive threshold calibration',
     'expected_gain': '+3-8% recall, better rho', 'effort': 'low',
     'implementation': 'Fit THRESH_REL per zarr based on intensity percentile gap at load time'},
    {'priority': 3, 'enhancement': 'Motion-model prediction (Kalman / constant-velocity)',
     'expected_gain': '+2-5% track precision', 'effort': 'medium',
     'implementation': 'Add Kalman filter to propagate position prediction between frames; use predicted pos as cost shift in Hungarian'},
    {'priority': 4, 'enhancement': 'Embryo-aware cross-attention linking',
     'expected_gain': '+3-6% edge Jaccard', 'effort': 'high',
     'implementation': 'Graph neural network (e.g. PointNet + attention) over per-embryo node sets'},
    {'priority': 5, 'enhancement': 'Test-time augmentation (TTA) for DL detector',
     'expected_gain': '+1-3% recall', 'effort': 'low',
     'implementation': 'Flip XY, average heatmaps before peak extraction; ~2x inference time'},
    {'priority': 6, 'enhancement': 'Multi-scale feature fusion in DL detector',
     'expected_gain': '+2-5% recall on dense regions', 'effort': 'medium',
     'implementation': 'Add ASPP / DeepLab-style multi-scale branch to TinyUNet3D'},
    {'priority': 7, 'enhancement': 'Active-learning annotation loop',
     'expected_gain': 'Better calibration on unseen embryos', 'effort': 'high',
     'implementation': 'Use uncertainty from MC-Dropout to flag frames for additional annotation'},
])
display(enhancement_df)
save_table(enhancement_df, 'phase10', 'enhancement_recommendations')

,priority,enhancement,expected_gain,effort,implementation
0,1,Larger DL model (UNet++ or 3D ResUNet),+5-15% recall,medium,Replace TinyUNet3D with 16-channel UNet++; tra...
1,2,Per-sample adaptive threshold calibration,"+3-8% recall, better rho",low,Fit THRESH_REL per zarr based on intensity per...
2,3,Motion-model prediction (Kalman / constant-vel...,+2-5% track precision,medium,Add Kalman filter to propagate position predic...
3,4,Embryo-aware cross-attention linking,+3-6% edge Jaccard,high,Graph neural network (e.g. PointNet + attentio...
4,5,Test-time augmentation (TTA) for DL detector,+1-3% recall,low,"Flip XY, average heatmaps before peak extracti..."
5,6,Multi-scale feature fusion in DL detector,+2-5% recall on dense regions,medium,Add ASPP / DeepLab-style multi-scale branch to...
6,7,Active-learning annotation loop,Better calibration on unseen embryos,high,Use uncertainty from MC-Dropout to flag frames...


PosixPath('/kaggle/working/outputs/tables/phase10_enhancement_recommendations.csv')

In [193]:
# ---- Figure 42: Enhancement priority vs expected gain ----

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))

# Map effort levels to numeric values
gain_map = {
    'low': 0.4,
    'medium': 0.65,
    'high': 0.90
}

effort_scores = [
    gain_map.get(str(e).lower(), 0.65)
    for e in enhancement_df['effort']
]

# Function to extract numeric gain values from text
def parse_gain(s):
    if pd.isna(s):
        return 5.0

    # Extract all numbers (supports integers and decimals)
    nums = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', str(s))]

    # Return the average if a range is found
    return np.mean(nums) if nums else 5.0

gain_scores = [
    parse_gain(s)
    for s in enhancement_df['expected_gain']
]

# Scatter plot
sc = ax.scatter(
    effort_scores,
    gain_scores,
    s=160,
    c=enhancement_df['priority'],
    cmap='RdYlGn_r',
    edgecolor='black',
    zorder=3
)

# Annotate each point
for i, lbl in enumerate(enhancement_df['enhancement']):
    ax.annotate(
        f"P{enhancement_df['priority'].iloc[i]}: {lbl[:30]}",
        (effort_scores[i], gain_scores[i]),
        fontsize=6.5,
        xytext=(5, 3),
        textcoords='offset points'
    )

# Labels and title
ax.set_xlabel('Effort (0 = Low, 1 = High)')
ax.set_ylabel('Expected Recall Gain (%)')
ax.set_title('Enhancement Roadmap: Gain vs. Effort')

# Colorbar
plt.colorbar(sc, ax=ax, label='Priority')

# Save figure
save_fig(fig, 'phase10', '42_enhancement_roadmap')

plt.tight_layout()
plt.show()

<Figure size 704x528 with 0 Axes>

In [197]:
# ---- Figure 43: hyperparameter importance bar (from calib sweep variance analysis) ----
fig, ax = plt.subplots(figsize=(7, 4))
if len(hp_sensitivity):
    ax.barh(hp_sensitivity['hyperparameter'], hp_sensitivity['recall_range'],
            xerr=hp_sensitivity['recall_std'], color=COLORS['blue'], edgecolor='k')
    ax.set_xlabel('recall range (max - min across values)'); ax.set_title('Hyperparameter recall sensitivity')
    ax.invert_yaxis()
else:
    ax.text(0.5, 0.5, 'no sweep data (needs real data)', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase10', '43_hyperparameter_importance')
plt.show()

In [196]:
# ---- Figure 44: per-embryo recall heatmap (embryo x sample) ----
fig, ax = plt.subplots(figsize=(8, 5))
if len(local_proxy_df) and 'sparse_recall' in local_proxy_df.columns:
    local_proxy_df['embryo'] = local_proxy_df['name'].apply(embryo_id)
    pivot = local_proxy_df.pivot_table(index='embryo', columns='name', values='sparse_recall', aggfunc='mean')
    im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns, rotation=60, fontsize=6)
    ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index)
    plt.colorbar(im, ax=ax, label='sparse recall')
    ax.set_title('Per-embryo sparse recall heatmap')
else:
    ax.text(0.5, 0.5, 'insufficient proxy data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase10', '44_embryo_recall_heatmap')
plt.show()

In [195]:
# ---- Figure 45: submission statistics summary violin / box ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
if len(sub_stats_df) and len(sub_stats_df) >= 2:
    axes[0].boxplot(sub_stats_df['n_nodes'].dropna(), patch_artist=True,
                     boxprops=dict(facecolor=COLORS['blue'], alpha=0.6))
    axes[0].set_title('Nodes per sample (submission)'); axes[0].set_ylabel('n_nodes')
    axes[1].boxplot(sub_stats_df['n_edges'].dropna(), patch_artist=True,
                     boxprops=dict(facecolor=COLORS['green'], alpha=0.6))
    axes[1].set_title('Edges per sample (submission)'); axes[1].set_ylabel('n_edges')
else:
    for ax in axes: ax.text(0.5, 0.5, 'insufficient submission data', ha='center', va='center'); ax.axis('off')
save_fig(fig, 'phase10', '45_submission_nodes_edges_boxplot')
plt.show()

In [194]:
# ---- 10.5  Overall run summary printout ----
summary_rows = []
summary_rows.append(['figures_saved', len(FIGURE_REGISTRY)])
summary_rows.append(['tables_saved', len(TABLE_REGISTRY)])
summary_rows.append(['dl_trained', DL_TRAINED])
summary_rows.append(['gpu_used', GPU_AVAILABLE])
summary_rows.append(['test_samples_processed', len(sub_stats_df)])
summary_rows.append(['total_submission_rows', running_id[0]])
summary_rows.append(['submission_file', str(SUBMISSION_CSV_PATH)])
summary_rows.append(['best_calib_recall', round(calib_summary['sparse_recall'].max(), 4) if len(calib_summary) and 'sparse_recall' in calib_summary else 'n/a'])
summary_rows.append(['active_detector', 'DL-CNN' if DL_TRAINED else 'Classical'])

run_summary_df = pd.DataFrame(summary_rows, columns=['item', 'value'])
display(run_summary_df)
save_table(run_summary_df, 'phase10', 'run_summary')

print('\n=== RUN COMPLETE ===')
print(f'Figures saved  : {len(FIGURE_REGISTRY)}')
print(f'Tables saved   : {len(TABLE_REGISTRY)}')
print(f'Active detector: {"DL-CNN" if DL_TRAINED else "Classical"}')
print(f'Test samples   : {len(sub_stats_df)}')
print(f'Submission rows: {running_id[0]}')
print(f'Submission CSV : {SUBMISSION_CSV_PATH}')

,item,value
0,figures_saved,43
1,tables_saved,16
2,dl_trained,False
3,gpu_used,True
4,test_samples_processed,4
5,total_submission_rows,86054
6,submission_file,/kaggle/working/outputs/submission/submission.csv
7,best_calib_recall,n/a
8,active_detector,Classical



=== RUN COMPLETE ===
Figures saved  : 43
Tables saved   : 17
Active detector: Classical
Test samples   : 4
Submission rows: 86054
Submission CSV : /kaggle/working/outputs/submission/submission.csv


## Phase 11 — Export & Packaging

All artefacts under `/kaggle/working/outputs/` are zipped into a single download.
The submission CSV is also written to `/kaggle/working/submission.csv` so Kaggle auto-commit picks it up.

In [198]:
# ---- Copy submission.csv to /kaggle/working/submission.csv (Kaggle expects it at root) ----
KAGGLE_SUB_PATH = Path('/kaggle/working/submission.csv')
if SUBMISSION_CSV_PATH.exists() and SUBMISSION_CSV_PATH != KAGGLE_SUB_PATH:
    shutil.copy(SUBMISSION_CSV_PATH, KAGGLE_SUB_PATH)
    print('Copied submission.csv to', KAGGLE_SUB_PATH)
elif SUBMISSION_CSV_PATH == KAGGLE_SUB_PATH:
    print('submission.csv already at', KAGGLE_SUB_PATH)
else:
    print('No submission.csv to copy (no test samples were processed).')

Copied submission.csv to /kaggle/working/submission.csv


In [199]:
# ---- Zip all outputs ----
ZIP_PATH = Path('/kaggle/working/all_outputs.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in OUT_ROOT.rglob('*'):
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(Path('/kaggle/working')))
print(f'All outputs zipped to: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)')

All outputs zipped to: /kaggle/working/all_outputs.zip  (3.0 MB)


In [200]:
# ---- Final registry printout ----
reg_df = pd.DataFrame(FIGURE_REGISTRY, columns=['phase', 'name', 'path'])
print(f'\n=== FIGURE REGISTRY: {len(reg_df)} figures ===')
display(reg_df)

tab_df = pd.DataFrame(TABLE_REGISTRY, columns=['phase', 'name', 'path'])
print(f'\n=== TABLE REGISTRY: {len(tab_df)} tables ===')
display(tab_df)

save_table(reg_df, 'phase11', 'figure_registry')
save_table(tab_df, 'phase11', 'table_registry')


=== FIGURE REGISTRY: 46 figures ===


,phase,name,path
0,phase2,01_split_embryo_counts,/kaggle/working/outputs/figures/phase2_01_spli...
1,phase2,02_array_geometry_hist,/kaggle/working/outputs/figures/phase2_02_arra...
2,phase2,03_estimated_nodes_hist,/kaggle/working/outputs/figures/phase2_03_esti...
3,phase2,04_estimated_nodes_per_frame,/kaggle/working/outputs/figures/phase2_04_esti...
4,phase3,05_edge_displacement_hist,/kaggle/working/outputs/figures/phase3_05_edge...
5,phase3,06_edge_dt_hist,/kaggle/working/outputs/figures/phase3_06_edge...
6,phase3,07_division_sister_distance,/kaggle/working/outputs/figures/phase3_07_divi...
7,phase3,08_label_ratio_per_sample,/kaggle/working/outputs/figures/phase3_08_labe...
8,phase3,09_divisions_and_node_edge_scatter,/kaggle/working/outputs/figures/phase3_09_divi...
9,phase3,10_gt_stats_by_embryo,/kaggle/working/outputs/figures/phase3_10_gt_s...



=== TABLE REGISTRY: 17 tables ===


,phase,name,path
0,phase0,gpu_diagnostics,/kaggle/working/outputs/tables/phase0_gpu_diag...
1,phase1,problem_contract,/kaggle/working/outputs/tables/phase1_problem_...
2,phase1,config_snapshot,/kaggle/working/outputs/tables/phase1_config_s...
3,phase2,split_summary,/kaggle/working/outputs/tables/phase2_split_su...
4,phase2,train_sample_metadata,/kaggle/working/outputs/tables/phase2_train_sa...
5,phase2,test_sample_metadata,/kaggle/working/outputs/tables/phase2_test_sam...
6,phase4,effective_tracking_geometry,/kaggle/working/outputs/tables/phase4_effectiv...
7,phase5,dl_training_history,/kaggle/working/outputs/tables/phase5_dl_train...
8,phase7,demo_track_nodes,/kaggle/working/outputs/tables/phase7_demo_tra...
9,phase7,demo_track_edges,/kaggle/working/outputs/tables/phase7_demo_tra...


PosixPath('/kaggle/working/outputs/tables/phase11_table_registry.csv')

In [201]:
# ---- Sanity check: count figures >= 40 ----
n_figs = len(FIGURE_REGISTRY)
n_tabs = len(TABLE_REGISTRY)
assert n_figs >= 40, f'Expected >=40 figures; got {n_figs}. Check earlier phases ran with data mounted.'
print(f'Sanity check PASSED: {n_figs} figures >= 40, {n_tabs} tables saved.')
print('Download all_outputs.zip from the output panel or via kaggle datasets API.')
print('submission.csv is at /kaggle/working/submission.csv and will be auto-committed by Kaggle.')

Sanity check PASSED: 46 figures >= 40, 19 tables saved.
Download all_outputs.zip from the output panel or via kaggle datasets API.
submission.csv is at /kaggle/working/submission.csv and will be auto-committed by Kaggle.


## Appendix — Notebook Map

| Phase | Key outputs |
|---|---|
| 0 | `phase0_gpu_diagnostics.csv` |
| 1 | `phase1_problem_contract.csv`, `phase1_config_snapshot.csv` |
| 2 | `phase2_split_summary.csv`, `phase2_train_sample_metadata.csv`, figs 01-04 |
| 3 | `phase3_gt_sample_stats.csv`, figs 05-10 |
| 4 | figs 11-16 (classical detector QA) |
| 5 | `phase5_dl_training_history.csv`, `models/tiny_unet3d_heatmap.pt`, figs 17-18 |
| 6 | `phase6_calibration_sweep_*.csv`, `phase6_permutation_feature_importance.csv`, `phase6_shap_*.csv`, `phase6_lime_*.csv`, figs 19-26 |
| 7 | `phase7_demo_track_nodes.csv`, `phase7_demo_track_edges.csv`, figs 27-29 |
| 8 | `phase8_local_proxy_scores.csv`, figs 30-31 |
| 9 | `submission/submission.csv`, `phase9_submission_audit.csv`, figs 32-35 |
| 10 | `phase10_model_comparison_table.csv`, `phase10_failure_mode_analysis.csv`, `phase10_enhancement_recommendations.csv`, `phase10_hyperparameter_sensitivity.csv`, figs 36-45 |
| 11 | `all_outputs.zip`, `phase11_figure_registry.csv` |